# ZIF-8 CO2/CH4 Separation: A Reproducibility-Graded Computational Pipeline

This notebook implements a complete computational pipeline for the analysis of CO2/CH4 separation using the metal-organic framework ZIF-8. The pipeline proceeds from raw data collection through reproducibility screening, Toth isotherm model fitting, Ideal Adsorbed Solution Theory (IAST) mixture predictions, and final publication-quality visualisation.

**Pipeline stages:**
1. Data Collection — NIST ISODB/MatDB API queries to retrieve ZIF-8 pure-component isotherms
2. Reproducibility Screening — Statistical outlier detection (Park et al. framework), IQR-based flagging, and R-grading
3. Consensus Isotherm Construction — Mean consensus with 95% confidence intervals from screened data
4. Isotherm Model Fitting — Toth model fitting; goodness-of-fit comparison against alternative models
5. IAST Predictions — Binary mixture loadings and selectivities via pressure and composition sweeps
6. Visualisation — Final publication-quality figures

**Adsorbent:** ZIF-8 (Zeolitic Imidazolate Framework-8)
**Adsorbates:** CO2 (Carbon Dioxide), CH4 (Methane)
**Data source:** NIST ISODB (https://adsorption.nist.gov/isodb)

## Section 1: Imports and Dependencies

In [ ]:
import os
import json
import math
import requests

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pathlib import Path
from scipy.optimize import curve_fit
from scipy.stats import t as t_dist
from sklearn.metrics import mean_squared_error, r2_score
from bs4 import BeautifulSoup

try:
    import ruptura
    RUPTURA_AVAILABLE = True
except ImportError:
    RUPTURA_AVAILABLE = False
    print("Note: ruptura is not installed. IAST sweep using ruptura (Section 5a) will be skipped.")

## Section 2: Data Collection

This section queries the NIST ISODB and associated local bibliography library to retrieve, filter, and organise ZIF-8 adsorption isotherms for CO2 and CH4. The workflow proceeds in four steps:

1. Query the NIST MatDB API to obtain a list of all ZIF materials with their unique hashkeys
2. Parse the local ISODB bibliography to extract isotherm-level metadata
3. Filter and validate isotherms (gas type, temperature window, experimental category)
4. Extract raw adsorption data and convert all units to mmol/g

### 2.1 Retrieve ZIF Material List from NIST MatDB

In [ ]:
url = "https://adsorption.nist.gov/matdb/content/plugins/matdb-search/functions.php?f=get_material_results"

payload = {
    "f": "get_material_results",
    "s": 0,
    "n": 1000,
    "q": "ZIF",
    "wid": "matdb"
}

headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "User-Agent": "Mozilla/5.0",
    "X-Requested-With": "XMLHttpRequest",
    "Origin": "https://adsorption.nist.gov",
    "Referer": "https://adsorption.nist.gov/matdb/index.php"
}

print("Sending request to NIST MatDB...")
resp = requests.post(url, data=payload, headers=headers)

if resp.status_code != 200:
    print(f"Request failed: HTTP {resp.status_code}")
else:
    resp_json = resp.json()
    if resp_json.get("status") != "Done":
        print(f"API error: {resp_json.get('message')}")
    else:
        raw_html = resp_json["o"]["output"]
        soup = BeautifulSoup(raw_html, "html.parser")

        records = []
        for li in soup.select("li.list-group-item"):
            name = li.get("data-name", "").strip()
            hashkey = li.get("data-id", "").strip()
            spans = li.find_all("span")
            if len(spans) >= 2:
                raw_syn = spans[1].decode_contents().replace("<br>", ",")
                synonyms = [s.strip() for s in raw_syn.split(",") if s.strip()]
            else:
                synonyms = []
            records.append({
                "name": name,
                "hashkey": hashkey,
                "synonyms": "; ".join(synonyms)
            })

        df_zif = pd.DataFrame(records)
        df_zif.to_csv("nist_ZIF_list.csv", index=False, encoding="utf-8-sig")
        print(f"Extracted and saved {len(df_zif)} ZIF entries to nist_ZIF_list.csv")

### 2.2 Parse ISODB Bibliography for Experimental Isotherms

In [ ]:
bibliography_dir = "../isodb-library/library/bibliography/"

print(f"Scanning bibliography directory: {bibliography_dir}")
json_files = [f for f in os.listdir(bibliography_dir) if f.endswith(".json")]
print(f"Found {len(json_files)} bibliography JSON files")

records = []

for idx, filename in enumerate(json_files, 1):
    path = os.path.join(bibliography_dir, filename)
    try:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        continue

    base = {
        "DOI": data.get("DOI"),
        "title": data.get("title"),
        "year": data.get("year"),
        "journal": data.get("journal"),
        "authors": ";".join(data.get("authors", [])),
        "categories": ";".join(data.get("categories", [])),
        "adsorbate_gases": ";".join(data.get("adsorbateGas", [])),
        "adsorbate_names": ";".join(a.get("name", "") for a in data.get("adsorbates", [])),
        "adsorbate_inchikeys": ";".join(a.get("InChIKey", "") for a in data.get("adsorbates", [])),
        "adsorbent_names": ";".join(a.get("name", "") for a in data.get("adsorbents", [])),
        "adsorbent_hashkeys": ";".join(a.get("hashkey", "") for a in data.get("adsorbents", [])),
        "temperatures": ";".join(str(t) for t in data.get("temperatures", [])),
        "pressures": ";".join(str(p) for p in data.get("pressures", []))
    }

    isotherms = data.get("isotherms", [])
    if not isotherms:
        continue

    for iso in isotherms:
        record = base.copy()
        record["isotherm_filename"] = iso.get("filename", "")
        records.append(record)

print(f"Parsed {len(records)} total isotherm records")
df_biblio = pd.DataFrame(records)

# Filter for experimental data and target gases only
df_biblio = df_biblio[df_biblio['categories'].fillna('').str.lower().str.contains('exp')]
target_gases = ['methane', 'carbon dioxide', 'nitrogen']
df_biblio = df_biblio[df_biblio['adsorbate_gases'].fillna('').str.lower().apply(
    lambda g: any(t in g for t in target_gases)
)]

df_biblio.to_csv("filtered_bibliography_for_CO2_CH4_N2_ZIFs.csv", index=False, encoding='utf-8-sig')
print(f"Filtered bibliography saved: {len(df_biblio)} rows")

### 2.3 Scan ISODB Library and Build Master Isotherm Metadata

In [ ]:
SOURCE_FOLDER = "../isodb-library/Library"
BIBLIO_CSV = "filtered_bibliography_for_CO2_CH4_N2_ZIFs.csv"
ZIF_CSV = "nist_ZIF_list.csv"

VALID_GASES = {"carbon dioxide", "methane", "nitrogen"}
TEMP_MIN, TEMP_MAX = 77, 303  # Kelvin - experimental window

print(f"Loading filtered bibliography: {BIBLIO_CSV}")
biblio_df = pd.read_csv(BIBLIO_CSV)
print(f"Starting with {len(biblio_df)} isotherm rows")

print(f"Loading ZIF list: {ZIF_CSV}")
zif_df = pd.read_csv(ZIF_CSV)
valid_hashkeys = set(zif_df["hashkey"].dropna())

valid_isofilenames = set(biblio_df["isotherm_filename"])
valid_folders = set(biblio_df["DOI"].str.replace("/", "", regex=False))

summary = []
total_scanned = 0
valid_found = 0

print(f"Scanning subfolders in: {SOURCE_FOLDER}")

for folder in os.listdir(SOURCE_FOLDER):
    if folder not in valid_folders:
        continue

    folder_path = os.path.join(SOURCE_FOLDER, folder)
    if not os.path.isdir(folder_path):
        continue

    for fname in os.listdir(folder_path):
        if not fname.endswith(".json"):
            continue

        total_scanned += 1
        fname_no_ext = fname.replace(".json", "")
        if fname_no_ext not in valid_isofilenames:
            continue

        full_path = os.path.join(folder_path, fname)
        try:
            with open(full_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"Failed to load {fname}: {e}")
            continue

        adsorbates = data.get("adsorbates", [])
        if len(adsorbates) == 1:
            adsorption_type = "single_component"
            gas = adsorbates[0].get("name", "").strip().lower()
            gas_inchikey = adsorbates[0].get("InChIKey", "")
        else:
            adsorption_type = "multi_component"
            gas = ";".join([a.get("name", "").strip().lower() for a in adsorbates])
            gas_inchikey = ";".join([a.get("InChIKey", "") for a in adsorbates])

        if gas not in VALID_GASES:
            continue

        adsorbent = data.get("adsorbent", {})
        hashkey = adsorbent.get("hashkey", "")
        adsorbent_name = adsorbent.get("name", "")
        if hashkey not in valid_hashkeys:
            continue

        temp = data.get("temperature")
        if temp is None or not (TEMP_MIN <= temp <= TEMP_MAX):
            continue

        isotherm_data = data.get("isotherm_data", [])
        num_points = len(isotherm_data)
        pressures_raw = [pt.get("pressure") for pt in isotherm_data if pt.get("pressure") is not None]
        adsorptions_raw = [pt.get("total_adsorption") for pt in isotherm_data if pt.get("total_adsorption") is not None]

        biblio_row = biblio_df[biblio_df["isotherm_filename"] == fname_no_ext]
        if biblio_row.empty:
            continue

        row = {f"paper_{k}": v for k, v in biblio_row.iloc[0].to_dict().items()}
        row.update({
            "json_filename": fname,
            "adsorption_type": adsorption_type,
            "adsorbate_gas": gas.upper(),
            "adsorbate_gas_inchikey": gas_inchikey,
            "temperature_K": temp,
            "pressure_units": data.get("pressureUnits", ""),
            "adsorption_units": data.get("adsorptionUnits", ""),
            "composition_type": data.get("compositionType", ""),
            "isotherm_type": data.get("isotherm_type", ""),
            "tabular_data_flag": data.get("tabular_data", ""),
            "date_recorded": data.get("date", ""),
            "adsorbent_name_from_json": adsorbent_name,
            "adsorbent_hashkey": hashkey,
            "num_points": num_points,
            "min_pressure": min(pressures_raw) if pressures_raw else None,
            "max_pressure": max(pressures_raw) if pressures_raw else None,
            "max_adsorption": max(adsorptions_raw) if adsorptions_raw else None
        })

        summary.append(row)
        valid_found += 1

print(f"Scan complete. Files scanned: {total_scanned}, valid matches: {valid_found}")

if summary:
    df_master = pd.DataFrame(summary)
    df_master.to_csv("isotherm_master_metadata.csv", index=False, encoding="utf-8-sig")
    print(f"Master metadata saved: {len(df_master)} rows -> isotherm_master_metadata.csv")
else:
    print("No valid isotherms matched all filters.")

### 2.4 Extract and Unit-Convert Isotherm Data for Manually Validated ZIF-8 Entries

In [ ]:
MASTER_METADATA_CSV = "isotherm_master_metadata_manual_vet.csv"
SOURCE_FOLDER = "../isodb-library/Library"
OUTPUT_ROOT = "isotherm_dataset_by_zif_gas_after_manual_validation"

GAS_MOLAR_VOLUMES = {
    "carbon dioxide": 22.414,  # L/mol at STP
    "methane": 22.414,
    "nitrogen": 22.414,
}
GAS_MOLAR_MASSES = {
    "carbon dioxide": 44.01,   # g/mol
    "methane": 16.04,
    "nitrogen": 28.014,
}

print(f"Loading master metadata: {MASTER_METADATA_CSV}")
meta_df = pd.read_csv(MASTER_METADATA_CSV)
print(f"Total isotherms in master: {len(meta_df)}")

# Apply manual validation filter: only include manually checked, valid-status entries
filtered_meta_df = meta_df[
    (meta_df["manual_check"].astype(str).str.lower() == 'true') &
    (meta_df["exp_status_check"].astype(str).str.lower() == 'valid')
]
print(f"Total isotherms after manual validation filter: {len(filtered_meta_df)}")

meta_lookup = {
    (row["paper_DOI"], row["json_filename"]): row
    for _, row in filtered_meta_df.iterrows()
}

all_keys = filtered_meta_df.groupby(["og_adsorbent", "adsorbate_gas"])

for (mof, gas), group in all_keys:
    print(f"Processing {mof} / {gas} ({len(group)} isotherms)")
    all_data = []

    for _, row in group.iterrows():
        doi = row["paper_DOI"]
        json_file = row["json_filename"]
        folder_name = doi.replace("/", "")
        full_path = os.path.join(SOURCE_FOLDER, folder_name, json_file)

        if (doi, json_file) not in meta_lookup:
            continue

        if not os.path.exists(full_path):
            print(f"Missing file: {full_path}")
            continue

        try:
            with open(full_path, "r", encoding="utf-8") as f:
                json_data = json.load(f)
        except Exception as e:
            print(f"Error loading {json_file}: {e}")
            continue

        adsorbates = json_data.get("adsorbates", [])
        if not adsorbates:
            continue

        gas_info = adsorbates[0]
        gas_name = gas_info.get("name", "").strip().lower()
        gas_inchikey = gas_info.get("InChIKey", "")
        adsorbent = json_data.get("adsorbent", {})
        adsorbent_name = adsorbent.get("name", "")
        unit_before = json_data.get("adsorptionUnits", "").strip().lower()
        molar_volume = GAS_MOLAR_VOLUMES.get(gas_name)
        molar_mass = GAS_MOLAR_MASSES.get(gas_name)

        isotherm_data = json_data.get("isotherm_data", [])
        for point in isotherm_data:
            pressure = point.get("pressure")
            adsorption = point.get("total_adsorption")
            if pressure is None or adsorption is None:
                continue

            # Convert adsorption to mmol/g
            if unit_before == "mmol/g":
                adsorption_mmol_g = adsorption
                unit_after = "mmol/g"
            elif unit_before in ("cm3(stp)/g", "ml/g") and molar_volume:
                adsorption_mmol_g = (adsorption / molar_volume) * 1000
                unit_after = "mmol/g"
            elif unit_before == "mg/g" and molar_mass:
                adsorption_mmol_g = adsorption / molar_mass
                unit_after = "mmol/g"
            elif unit_before == "wt%" and molar_mass:
                adsorption_mmol_g = (adsorption / 100) * (1000 / molar_mass)
                unit_after = "mmol/g"
            else:
                adsorption_mmol_g = math.nan
                unit_after = "unknown"

            meta = meta_lookup.get((doi, json_file), {})
            all_data.append({
                "paper_DOI": doi,
                "paper_categories": meta.get("paper_categories", ""),
                "paper_title": meta.get("paper_title", ""),
                "paper_journal": meta.get("paper_journal", ""),
                "paper_authors": meta.get("paper_authors", ""),
                "paper_year": meta.get("paper_year", ""),
                "isotherm_json": json_file,
                "adsorbent": adsorbent_name,
                "adsorbent_hashkey": adsorbent.get("hashkey", ""),
                "adsorbate": gas_name.upper(),
                "adsorbate_inchikey": gas_inchikey,
                "temperature_K": json_data.get("temperature"),
                "composition_type": json_data.get("compositionType", ""),
                "pressure_units": json_data.get("pressureUnits", ""),
                "adsorption_units_before": unit_before,
                "adsorption": adsorption,
                "adsorption_mmol_per_g": adsorption_mmol_g,
                "adsorption_units_after": unit_after,
                "pressure": pressure
            })

    if all_data:
        df_out = pd.DataFrame(all_data)
        safe_mof = mof.replace("/", "_").replace("\\", "_")
        safe_gas = gas.replace("/", "_").replace("\\", "_")
        folder_out = os.path.join(OUTPUT_ROOT, safe_mof)
        os.makedirs(folder_out, exist_ok=True)
        file_out = os.path.join(folder_out, f"{safe_mof}__{safe_gas}.csv")
        df_out.to_csv(file_out, index=False, encoding="utf-8")
        print(f"Saved {len(df_out)} rows to {file_out}")
    else:
        print(f"No valid data found for {mof} / {gas}.")

## Section 3: Reproducibility Screening

This section implements the Park et al. reproducibility framework applied independently to ZIF-8 methane and carbon dioxide isotherms. The pipeline performs:

1. Toth model fitting to each individual isotherm on a common log-spaced pressure grid
2. Point-wise outlier detection using adaptive methods (Tukey IQR for N > 4; RMSE/sigma for N = 3 or 4)
3. Isotherm-level outlier discarding (flag threshold: more than 4 flagged pressure points)
4. Reproducibility grading (R1 through R4) based on the outlier fraction
5. Consensus isotherm construction (mean + 95% confidence interval) from retained isotherms
6. Output of consensus CSV files and reproducibility plots

### 3.1 Toth Isotherm Model Definitions

In [ ]:
def toth_model_full(p, Q_m, K_t, t):
    """
    Three-parameter Toth isotherm model.

    Parameters
    ----------
    p : array-like
        Pressure values (bar).
    Q_m : float
        Monolayer capacity (mmol/g).
    K_t : float
        Toth affinity constant (1/bar).
    t : float
        Toth heterogeneity exponent (dimensionless).

    Returns
    -------
    array-like
        Predicted adsorption (mmol/g).
    """
    term = K_t * p
    denominator_term = np.power(1 + np.power(term, t), 1.0 / t)
    return Q_m * K_t * p / (denominator_term + np.finfo(float).eps)


def toth_model_fixed_Qm(p, K_t, t, fixed_Qm_val):
    """
    Two-parameter Toth isotherm model with Qm fixed to a physical upper bound.

    Parameters
    ----------
    p : array-like
        Pressure values (bar).
    K_t : float
        Toth affinity constant (1/bar).
    t : float
        Toth heterogeneity exponent (dimensionless).
    fixed_Qm_val : float
        Pre-specified monolayer capacity (mmol/g).

    Returns
    -------
    array-like
        Predicted adsorption (mmol/g).
    """
    term = K_t * p
    denominator_term = np.power(1 + np.power(term, t), 1.0 / t)
    return fixed_Qm_val * K_t * p / (denominator_term + np.finfo(float).eps)

### 3.2 Reproducibility Screening - Methane

In [ ]:
# =========================================================================
# PARK REPRODUCIBILITY FRAMEWORK (FULL IMPLEMENTATION FOR METHANE)
# =========================================================================
# This script performs the full Park et al. reproducibility analysis for
# ZIF-8 with Carbon Dioxide, including dynamic outlier detection, mean-based
# consensus, and plots results using matplotlib with specific axis scaling.

# === CONFIGURATION PARAMETERS ===
INPUT_CSV = "./zif-code/isotherm_dataset_by_zif_gas_after_manual_validation/ZIF-8/ZIF-8__Methane.csv" 
OUTPUT_DIR = "park_reproducibility_output_final/methane"  
current_gas_name = 'Methane' # Define current_gas_name for consistent output messages

# --- Common Pressure Points (Log-spaced for granularity) ---
log_min_p = np.log10(0.01) # log10(0.01) = -2
log_max_p = np.log10(60.0) # log10(60) approx = 1.778

N_POINTS = 50 # Total number of evaluation points for the log-spaced grid
COMMON_PRESSURE_POINTS = np.logspace(log_min_p, log_max_p, N_POINTS)

# --- Physical Qm Limits (GLOBAL, then used for specific gas) ---
MAX_PHYSICAL_QM_METHANE = 7.0 # mmol/g for CH4 on ZIF-8 (from literature/pore volume estimates)
MAX_PHYSICAL_QM_CO2 = 9.0     # mmol/g for CO2 on ZIF-8 (from literature/pore volume estimates)

# --- Determine MAX_PHYSICAL_QM for the current gas ---
MAX_PHYSICAL_QM_FOR_CURRENT_GAS = MAX_PHYSICAL_QM_METHANE

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)





# === STEP 1: GROUP DATA BY INDIVIDUAL ISOTHERMS AND STORE PRESSURE RANGES ===
print("=== STEP 1: GROUPING ISOTHERMS AND RECORDING RANGES ===")
df = pd.read_csv(INPUT_CSV)
grouped = df.groupby("isotherm_json")
print(f"Found {len(grouped)} unique isotherms in the dataset")

isotherm_pressure_ranges = {}
isotherm_data_distribution = {}

for isotherm_name, group_data in grouped:
    pressures = group_data["pressure"].dropna()
    adsorptions = group_data["adsorption_mmol_per_g"].dropna()

    isotherm_data_distribution[isotherm_name] = {
        "num_points": len(group_data),
        "min_pressure": pressures.min() if not pressures.empty else np.nan,
        "max_pressure": pressures.max() if not pressures.empty else np.nan,
        "min_adsorption": adsorptions.min() if not adsorptions.empty else np.nan,
        "max_adsorption": adsorptions.max() if not adsorptions.empty else np.nan,
    }

    if not pressures.empty and len(pressures) >= 2:
        isotherm_pressure_ranges[isotherm_name] = (pressures.min(), pressures.max())
        print(f"\n--- Processing Isotherm: {isotherm_name} ---")
        print(f"  Original data points: {isotherm_data_distribution[isotherm_name]['num_points']}")
        print(f"  Original pressure range: [{isotherm_data_distribution[isotherm_name]['min_pressure']:.2f}, {isotherm_data_distribution[isotherm_name]['max_pressure']:.2f}] bar")
        print(f"  Original adsorption range: [{isotherm_data_distribution[isotherm_name]['min_adsorption']:.2f}, {isotherm_data_distribution[isotherm_name]['max_adsorption']:.2f}] mmol/g")
    else:
        print(f"\n--- Skipping Isotherm: {isotherm_name} ---")
        print(f"   Insufficient original data points ({len(pressures)}) to determine pressure range (min 2 required).")

# === STEP 2: CREATE GLOBAL INTERPOLATION GRID ===
print("\n=== STEP 2: CREATING GLOBAL INTERPOLATION GRID ===")
pressure_grid = COMMON_PRESSURE_POINTS
print(f"Created global interpolation grid with {N_POINTS} pressure points from {pressure_grid.min():.2f} to {pressure_grid.max():.2f} bar.")
print(f"Global pressure grid points: {np.array2string(pressure_grid, precision=2, separator=', ')}")

# === STEP 3: FIT TOTH MODEL AND EVALUATE ISOTHERMS (ENHANCED DIAGNOSTICS & CONDITIONAL Qm) ===
print("\n=== STEP 3: FITTING TOTH MODEL AND EVALUATING ISOTHERMS (ENHANCED DIAGNOSTICS & CONDITIONAL Qm) ===")
interp_data = {}
isotherm_fit_status = {}
fitted_params_summary = {} 

for isotherm_id, group in grouped:
    print(f"\n--- Toth Fitting for: {isotherm_id} ---")
    p_exp = group["pressure"].values
    q_exp = group["adsorption_mmol_per_g"].values

    valid_indices = ~np.isnan(p_exp) & ~np.isnan(q_exp)
    p_exp_clean = p_exp[valid_indices]
    q_exp_clean = q_exp[valid_indices]

    print(f"  Clean data points for fitting: {len(p_exp_clean)}")
    if len(p_exp_clean) > 0:
        print(f"    Sample clean pressures (first 5): {np.array2string(p_exp_clean[:5], precision=4, separator=', ')}")
        print(f"    Sample clean adsorptions (first 5): {np.array2string(q_exp_clean[:5], precision=4, separator=', ')}")
        if len(p_exp_clean) > 5:
            print(f"    Sample clean pressures (last 5): {np.array2string(p_exp_clean[-5:], precision=4, separator=', ')}")
            print(f"    Sample clean adsorptions (last 5): {np.array2string(q_exp_clean[-5:], precision=4, separator=', ')}")

    if isotherm_id not in isotherm_pressure_ranges:
        print(f"   Skipping Toth fit: No valid pressure range recorded in Step 1.")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Skipped_NoOriginalRange"
        continue

    if len(p_exp_clean) < 3:
        print(f"   Skipping Toth fit: Insufficient clean data points ({len(p_exp_clean)}) for Toth model fitting (min 3 required).")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Skipped_InsufficientData"
        continue
    
    # Heuristic for deciding whether to fix Qm
    if np.max(q_exp_clean) < (MAX_PHYSICAL_QM_FOR_CURRENT_GAS / 2.0):
        use_fixed_Qm = True
        fixed_Qm_val_for_fit = MAX_PHYSICAL_QM_FOR_CURRENT_GAS
        print(f"  Deciding to FIX Qm: Max adsorption ({np.max(q_exp_clean):.2f}) is low, fitting with fixed Qm={fixed_Qm_val_for_fit:.2f}.")
    else:
        use_fixed_Qm = False
        print(f"  Deciding to FIT Qm: Max adsorption ({np.max(q_exp_clean):.2f}) is high enough.")


    try:
        if use_fixed_Qm:
            initial_guesses = [1.0, 1.0] # K_t, t
            # bounds = ([1e-6, 1e-6], [np.inf, 2.0]) # K_t bounds, t bounds (t up to 2.0)
            bounds = ([1e-6, 1e-6], [np.inf, 1.1]) # K_t bounds, t bounds (t up to 2.0)
            
            params, covariance = curve_fit(lambda p, K_t, t: toth_model_fixed_Qm(p, K_t, t, fixed_Qm_val_for_fit),
                                           p_exp_clean, q_exp_clean,
                                           p0=initial_guesses, bounds=bounds, maxfev=20000, 
                                           ftol=1e-8, xtol=1e-8, gtol=1e-8)
            K_t_fit, t_fit = params
            Q_m_fit = fixed_Qm_val_for_fit # Record the fixed Qm
            print(f"   Toth Fit (Qm FIXED) Successful!")
            print(f"    Fitted Parameters: Q_m={Q_m_fit:.4f} (FIXED), K_t={K_t_fit:.4f}, t={t_fit:.4f}")
            fitted_params_summary[isotherm_id] = {'Q_m': Q_m_fit, 'K_t': K_t_fit, 't': t_fit, 'fit_type': 'Fixed_Qm'}

        else:
            initial_guesses = [np.max(q_exp_clean) * 1.2, 1.0, 1.0]
            if initial_guesses[0] <= 0 or np.isnan(initial_guesses[0]):
                initial_guesses[0] = 0.1
            # bounds = ([1e-6, 1e-6, 1e-6], [np.inf, np.inf, 2.0])
            bounds = ([1e-6, 1e-6, 1e-6], [np.inf, np.inf, 1.1])
            
            params, covariance = curve_fit(toth_model_full, p_exp_clean, q_exp_clean,
                                           p0=initial_guesses, bounds=bounds, maxfev=20000, 
                                           ftol=1e-8, xtol=1e-8, gtol=1e-8)
            Q_m_fit, K_t_fit, t_fit = params
            print(f"   Toth Fit (Full) Successful!")
            print(f"    Fitted Parameters: Q_m={Q_m_fit:.4f}, K_t={K_t_fit:.4f}, t={t_fit:.4f}")
            fitted_params_summary[isotherm_id] = {'Q_m': Q_m_fit, 'K_t': K_t_fit, 't': t_fit, 'fit_type': 'Full_Fit'}

        # Check if parameters hit bounds 
        kt_lower_bound_val = bounds[0][0] if use_fixed_Qm else bounds[0][1]
        kt_upper_bound_val = bounds[1][0] if use_fixed_Qm else bounds[1][1]
        t_lower_bound_val = bounds[0][1] if use_fixed_Qm else bounds[0][2]
        t_upper_bound_val = bounds[1][1] if use_fixed_Qm else bounds[1][2]

        if np.isclose(K_t_fit, kt_lower_bound_val) or np.isclose(K_t_fit, kt_upper_bound_val):
            print(f"     Warning: K_t ({K_t_fit:.4f}) hit a bound.")
        
        if np.isclose(t_fit, t_lower_bound_val) or np.isclose(t_fit, t_upper_bound_val):
            print(f"     Warning: t ({t_fit:.4f}) hit a bound.")
        
        if not use_fixed_Qm and (np.isclose(Q_m_fit, bounds[0][0]) or np.isclose(Q_m_fit, bounds[1][0])):
             print(f"     Warning: Q_m ({Q_m_fit:.4f}) hit a bound.")
        if not use_fixed_Qm and Q_m_fit > MAX_PHYSICAL_QM_FOR_CURRENT_GAS * 1.5: 
            print(f"     Warning: Fitted Qm ({Q_m_fit:.2f}) is significantly above physical limit ({MAX_PHYSICAL_QM_FOR_CURRENT_GAS:.2f}).")


        interp_q = toth_model_full(pressure_grid, Q_m_fit, K_t_fit, t_fit)

        min_original_p_val, max_original_p_val = isotherm_pressure_ranges[isotherm_id]
        mask_above_max = (pressure_grid > max_original_p_val)
        
        interp_q[mask_above_max] = np.nan
        
        interp_data[isotherm_id] = interp_q
        isotherm_fit_status[isotherm_id] = "Fit_Success"
        print(f"   Evaluated on global grid. Values ABOVE original range masked with NaN (allowing extrapolation below min).")
        sample_indices = [0, len(pressure_grid)//4, len(pressure_grid)//2, 3*len(pressure_grid)//4, len(pressure_grid)-1]
        sample_pressures = pressure_grid[sample_indices]
        sample_adsorptions = interp_q[sample_indices]
        print(f"    Sample evaluated P (bar): {np.array2string(sample_pressures, precision=2, separator=', ')}")
        print(f"    Sample evaluated Q (mmol/g): {np.array2string(sample_adsorptions, precision=4, separator=', ')}")


    except RuntimeError as e:
        print(f"   Toth Fit FAILED: {e}. (Could not converge or find optimal parameters)")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Fit_Failed_Runtime"
    except ValueError as e:
        print(f"   Toth Fit FAILED: {e}. (Invalid parameters or data encountered, e.g., all clean_q_exp is zero or negative)")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Fit_Failed_Value"
    except Exception as e:
        print(f"   Toth Fit FAILED: An unexpected error occurred: {e}")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Fit_Failed_Other"

print(f"\n=== Toth Model Fitting Stage Complete ===")
print(f"Status for all isotherms:")
for iso_id, status in isotherm_fit_status.items():
  print(f" - {iso_id}: {status}")

# --- Display DataFrame of all Toth-fitted isotherms ---
print(f"\n=== Toth Fitted Isotherms Data (mmol/g) ===")
df_display_data = {
  iso_id: interp_data[iso_id] for iso_id in sorted(interp_data.keys())
}
df_toth_fitted = pd.DataFrame(df_display_data, index=pressure_grid)
df_toth_fitted.index.name = "Pressure (bar)"
print(df_toth_fitted.to_string(float_format="%.4f"))


# =========================================================================
# STEP 4: OUTLIER DETECTION (FULL IMPLEMENTATION AS PER SUPERVISOR DIRECTIVE)
# =========================================================================
print(f"\n=== STEP 4: OUTLIER DETECTION ===")

eligible_isotherm_ids = [iso_id for iso_id, status in isotherm_fit_status.items() if status == "Fit_Success"]
eligible_matrix = np.array([interp_data[iso_id] for iso_id in eligible_isotherm_ids])
N_eligible = len(eligible_isotherm_ids)

print(f"Analyzing {N_eligible} eligible isotherms for outliers...")

outlier_flags = {iso_id: "Not_Analyzed" for iso_id in df.groupby("isotherm_json").groups.keys()}
isotherm_flagged_points_count = {iso_id: 0 for iso_id in eligible_isotherm_ids} # Reset for a clean count this run
isotherm_rmse_values = {iso_id: np.nan for iso_id in eligible_isotherm_ids} # For O2 method only

print("\n Outlier detection results per pressure point:")
for j, p_val in enumerate(pressure_grid):
  values_at_pressure = eligible_matrix[:, j]
  values_at_pressure_clean = values_at_pressure[~np.isnan(values_at_pressure)]
  
  N_local = len(values_at_pressure_clean)
  
  # Collect flags for this pressure point
  flags_at_this_point = [] 

  if N_local < 2:
    print(f"  P={p_val:.2f} bar: {N_local} isotherms contribute. Skipping outlier check (N_local < 2).")
    continue 

  # Determine method based on N_local
  if N_local > 4: # O1 Method (Tukey's Rule)
    q1 = np.percentile(values_at_pressure_clean, 25)
    q3 = np.percentile(values_at_pressure_clean, 75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    for i_iso, iso_id in enumerate(eligible_isotherm_ids):
      current_value = eligible_matrix[i_iso, j]
      if not np.isnan(current_value):
        if current_value < lower_bound or current_value > upper_bound:
          flags_at_this_point.append(f" {iso_id} (Val={current_value:.2f}, Bounds=[{lower_bound:.2f}, {upper_bound:.2f}], Method: O1)")
          isotherm_flagged_points_count[iso_id] += 1
  
  elif N_local in [3, 4]: # O2 Method (RMSE/sigma comparison)
    median_uptake = np.nanmedian(values_at_pressure_clean)
    std_uptake = np.nanstd(values_at_pressure_clean)
    
    if np.isnan(std_uptake) or std_uptake == 0:
      print(f"    Cannot apply O2 method at P={p_val:.2f} due to zero/NaN std_uptake.")
      continue

    potential_flags_at_this_point = []
    for i_iso, iso_id in enumerate(eligible_isotherm_ids):
      current_value = eligible_matrix[i_iso, j]
      if not np.isnan(current_value):
        rmse_point = np.sqrt(mean_squared_error([median_uptake], [current_value]))
        
        if rmse_point > std_uptake / 2:
          potential_flags_at_this_point.append(iso_id)
    
    # --- NEW ROBUSTNESS CHECK FOR N_LOCAL = 3 or 4 ---
    if len(potential_flags_at_this_point) > 1: # If 2 or more would be flagged
      print(f"    P={p_val:.2f}: N_local={N_local}. {len(potential_flags_at_this_point)} isotherms potentially flagged. (High spread, no distinct outliers at this point).")
    else: # If 0 or 1 isotherm is flagged, then apply the flag.
      for iso_id_flagged in potential_flags_at_this_point:
        isotherm_flagged_points_count[iso_id_flagged] += 1
        print(f"    {iso_id_flagged} FLAGGED at P={p_val:.2f} (RMSE={rmse_point:.2f}, Std/2={std_uptake/2:.2f}). Method: O2")
  # Print summary for this pressure point
  if flags_at_this_point:
    print(f"  P={p_val:.2f} bar: {N_local} isotherms contribute. Flags: {' '.join(flags_at_this_point)}")
  else:
    print(f"  P={p_val:.2f} bar: {N_local} isotherms contribute. No outliers found at this point.")

print("\n Applying supervisor's outlier discard rule:")
for iso_id in eligible_isotherm_ids:
  flagged_count = isotherm_flagged_points_count[iso_id]
  
  if flagged_count <= 4:
    outlier_flags[iso_id] = "OK"
    print(f"   {iso_id}: Flagged at {flagged_count} points. KEPT (OK).")
  else:
    outlier_flags[iso_id] = "Outlier_Discarded"
    print(f"   {iso_id}: Flagged at {flagged_count} points. DISCARDED (Outlier).")

N_prime = sum(1 for flag in outlier_flags.values() if flag == "OK")
outlier_count_final = sum(1 for flag in outlier_flags.values() if flag == "Outlier_Discarded")
print(f"Total {N_prime} isotherms are OK for consensus.")
print(f"Total {outlier_count_final} isotherms are discarded as outliers.")


# === STEP 5: ASSIGN REPRODUCIBILITY GRADE ===
print(f"\n=== STEP 5: ASSIGN REPRODUCIBILITY GRADING ===")

if N_eligible > 2:
  f = (N_eligible - N_prime) / N_eligible
  print(f"Outlier fraction (f): {f:.2f}")
else:
  f = None
  print(f"Outlier fraction (f): Not applicable due to N_eligible < 3.")


if N_eligible > 4:
  if f is not None and f <= 0.25:
    reproducibility = "R1"
    grade_explanation = "Excellent: 25% outliers"
  elif f is not None and f <= 0.4:
    reproducibility = "R2"
    grade_explanation = "Good: 25-40% outliers"
  else:
    reproducibility = "R3"
    grade_explanation = "Moderate: >40% outliers"

elif N_eligible in [3, 4]:
  reproducibility = "R2"
  grade_explanation = "Good: 3-4 eligible isotherms available"

elif N_eligible == 2:
  curve1 = eligible_matrix[0]
  curve2 = eligible_matrix[1]
  common_valid_points = ~np.isnan(curve1) & ~np.isnan(curve2)
  
  if np.sum(common_valid_points) < 1:
    reproducibility = "Insufficient data"
    grade_explanation = "Cannot assess with unreliable RMSE/sigma for 2 isotherms (no common points)"
  else:
    rmse_between_curves = np.sqrt(mean_squared_error(curve1[common_valid_points], curve2[common_valid_points]))
    combined_matrix_for_std = np.array([curve1[common_valid_points], curve2[common_valid_points]])
    sigma_overall = np.mean(np.std(combined_matrix_for_std, axis=0))
    
    if rmse_between_curves <= sigma_overall / 2:
      reproducibility = "R3"
      grade_explanation = "Moderate: Low spread between 2 isotherms"
    else:
      reproducibility = "R4"
      grade_explanation = "Poor: High spread between 2 isotherms"

else: # N_eligible < 2
  reproducibility = "Insufficient data"
  grade_explanation = "Cannot assess with <2 eligible isotherms"

print(f"Reproducibility Grade: {reproducibility}")
print(f"Explanation: {grade_explanation}")


# =========================================================================
# STEP 6: CREATE CONSENSUS ISOTHERM (Mean uptake & Standard Deviation)
# =========================================================================
print(f"\n=== STEP 6: CONSENSUS ISOTHERM GENERATION ===")

valid_curves_for_consensus = []
for iso_id in eligible_isotherm_ids:
  if outlier_flags.get(iso_id) == "OK":
    valid_curves_for_consensus.append(interp_data[iso_id])
    print(f"Including {iso_id} in consensus (OK status).")
  else:
    print(f"Excluding {iso_id} from consensus (flagged: {outlier_flags.get(iso_id, 'Not OK')}).")

if valid_curves_for_consensus:
  # Calculate initial consensus mean and std from OK isotherms only
  consensus_mean = np.nanmean(valid_curves_for_consensus, axis=0)
  consensus_std = np.nanstd(valid_curves_for_consensus, axis=0)
  print(f"Consensus based on {len(valid_curves_for_consensus)} OK isotherms.")
else:
  # Fallback for if NO isotherms were OK AT ALL
  print(" No isotherms marked as 'OK' for consensus. Calculating consensus (mean & std) from ALL eligible isotherms as fallback.")
  consensus_mean = np.nanmean(eligible_matrix, axis=0)
  consensus_std = np.nanstd(eligible_matrix, axis=0)
  print(f"Consensus based on all {N_eligible} eligible isotherms (no 'OK' exclusions).")

# --- NEW: Hybrid Consensus Logic (Fill NaNs with all eligible data) ---
# This loop will fill NaNs in consensus_mean and consensus_std if they occurred due to sparsity
# even when 'OK' isotherms existed at other pressures.
# It will use *all eligible isotherms* for those specific points where primary consensus is NaN.
consensus_mean_hybrid = consensus_mean.copy()
consensus_std_hybrid = consensus_std.copy()
N_local_consensus = np.full(N_POINTS, np.nan) # To store N_local for CI calculation

for j in range(N_POINTS):
  current_values_ok = np.array([curve[j] for curve in valid_curves_for_consensus if not np.isnan(curve[j])])
  
  if len(current_values_ok) > 0:
    N_local_consensus[j] = len(current_values_ok)
  else: # If primary consensus is NaN at this point, use all eligible data
    values_at_pressure_all_eligible = eligible_matrix[:, j]
    values_at_pressure_all_eligible_clean = values_at_pressure_all_eligible[~np.isnan(values_at_pressure_all_eligible)]

    if len(values_at_pressure_all_eligible_clean) > 0:
      consensus_mean_hybrid[j] = np.nanmean(values_at_pressure_all_eligible_clean)
      consensus_std_hybrid[j] = np.nanstd(values_at_pressure_all_eligible_clean)
      N_local_consensus[j] = len(values_at_pressure_all_eligible_clean)
      print(f" P={pressure_grid[j]:.2f} bar: Consensus was NaN. Filled using mean/std of {len(values_at_pressure_all_eligible_clean)} eligible isotherms (including discarded).")
    else:
      N_local_consensus[j] = 0 # No data even from all eligible

# Reassign consensus variables to the hybrid version for final output
consensus_mean = consensus_mean_hybrid
consensus_std = consensus_std_hybrid

# --- Calculate 95% Confidence Interval ---
confidence_level = 0.95
alpha = 1.0 - confidence_level

consensus_sem = np.full(N_POINTS, np.nan)
consensus_ci_lower = np.full(N_POINTS, np.nan)
consensus_ci_upper = np.full(N_POINTS, np.nan)

for j in range(N_POINTS):
  n = N_local_consensus[j] # Use the N_local that contributed to the final consensus_mean at this point
  current_std = consensus_std[j] # Use the hybrid std for CI calculation
  current_mean = consensus_mean[j] # Use the hybrid mean

  if n > 1 and not np.isnan(current_std): # Need at least 2 points for std dev, and n-1 degrees of freedom for t-dist
    sem = current_std / np.sqrt(n)
    degrees_freedom = n - 1
    t_score = t_dist.ppf(1 - alpha / 2, degrees_freedom) # Two-tailed t-score
    
    consensus_sem[j] = sem
    consensus_ci_lower[j] = current_mean - t_score * sem
    consensus_ci_upper[j] = current_mean + t_score * sem
  elif n == 1 and not np.isnan(current_std): # If only one point, SD is 0, SEM is 0. CI is just the point itself.
    consensus_sem[j] = 0.0 # SEM is effectively 0 for N=1
    consensus_ci_lower[j] = current_mean
    consensus_ci_upper[j] = current_mean
  # If n=0 or nan, CI remains nan.

# If all values in consensus_mean are NaN (e.g., no data points at all)
if np.all(np.isnan(consensus_mean)):
  print("Warning: Consensus mean is all NaNs. This might indicate no data points at any pressure.")
  consensus_mean = np.zeros(N_POINTS)
  consensus_std = np.zeros(N_POINTS)
  consensus_sem = np.zeros(N_POINTS)
  consensus_ci_lower = np.zeros(N_POINTS)
  consensus_ci_upper = np.zeros(N_POINTS)

# Adjust consensus_plus_sd / consensus_minus_sd to represent CI
consensus_plus_sd = consensus_ci_upper # Now represents upper CI bound
consensus_minus_sd = consensus_ci_lower # Now represents lower CI bound


# =========================================================================
# STEP 7: SAVE OUTPUTS & PLOT (MATPLOTLIB)
# =========================================================================
print(f"\n=== STEP 7: SAVING RESULTS & PLOTTING ===")

# === SAVE CONSENSUS ISOTHERM CSV ===
consensus_df = pd.DataFrame({
  "pressure_bar": pressure_grid,
  "consensus_adsorption_mean_mmol_g": consensus_mean,
  "consensus_adsorption_std_mmol_g": consensus_std,
  "consensus_adsorption_sem_mmol_g": consensus_sem, # NEW
  "consensus_adsorption_ci_lower_mmol_g": consensus_ci_lower, # NEW
  "consensus_adsorption_ci_upper_mmol_g": consensus_ci_upper, # NEW
})
consensus_filename = os.path.join(OUTPUT_DIR, f"ZIF-8__{current_gas_name.replace(' ', '_')}_consensus_isotherm.csv") # Fixed filename for saving
consensus_df.to_csv(consensus_filename, index=False)
print(f" Saved consensus isotherm (mean and std): {consensus_filename}")

# --- NEW: Save All Individual Isotherms + Consensus to One CSV ---
combined_output_df = df_toth_fitted.copy() # Start with the Toth-fitted individual isotherms
combined_output_df['Consensus_Mean_mmol_g'] = consensus_mean
combined_output_df['Consensus_Std_mmol_g'] = consensus_std
combined_output_df['Consensus_SEM_mmol_g'] = consensus_sem # NEW
combined_output_df['Consensus_CI_Lower_mmol_g'] = consensus_ci_lower # NEW
combined_output_df['Consensus_CI_Upper_mmol_g'] = consensus_ci_upper # NEW

combined_output_filename = os.path.join(OUTPUT_DIR, f"ZIF-8__{current_gas_name.replace(' ', '_')}_all_isotherms_and_consensus.csv") # Fixed filename for saving
combined_output_df.to_csv(combined_output_filename, index=True) # index=True to include Pressure (bar)
print(f" Saved all Toth-fitted isotherms and consensus to: {combined_output_filename}")


summary_rows = []
for iso_id in df.groupby("isotherm_json").groups.keys(): # Iterate over all original isotherms
  summary_rows.append({
    "isotherm_json": iso_id,
    "fit_status": isotherm_fit_status.get(iso_id, "Unknown"),
    "outlier_final_status": outlier_flags.get(iso_id, "Not_Analyzed"),
    "flagged_points_count": isotherm_flagged_points_count.get(iso_id, np.nan) if isotherm_fit_status.get(iso_id) == "Fit_Success" else np.nan,
    "reproducibility_grade": reproducibility # Global grade applies to all in summary
  })

summary_df = pd.DataFrame(summary_rows)
summary_filename = os.path.join(OUTPUT_DIR, f"ZIF-8__{current_gas_name.replace(' ', '_')}_isotherm_summary.csv") # Fixed filename for saving
summary_df.to_csv(summary_filename, index=False)
print(f" Saved summary table: {summary_filename}")


# === CREATE VISUALIZATION (MATPLOTLIB) ===
print("Creating visualization (matplotlib)...")
fig, ax_main = plt.subplots(figsize=(10, 6))

cmap = plt.cm.get_cmap('tab10', len(df.groupby("isotherm_json").groups.keys()))
colors_list = [cmap(i) for i in range(len(df.groupby("isotherm_json").groups.keys()))]
iso_color_map = {iso_id: colors_list[i] for i, iso_id in enumerate(df.groupby("isotherm_json").groups.keys())}


for iso_id in df.groupby("isotherm_json").groups.keys():
  y = interp_data[iso_id]
  
  current_outlier_final_status = outlier_flags.get(iso_id, "Not_Analyzed")
  current_fit_status = isotherm_fit_status.get(iso_id, "Unknown")

  label_suffix = ""
  line_style = '-'
  alpha_val = 0.8
  color_val = iso_color_map.get(iso_id, 'black')
  marker_style = 'o'
  line_width = 1.5

  if current_fit_status != "Fit_Success":
    line_style = ':'
    alpha_val = 0.3
    color_val = 'grey'
    marker_style = 'x'
    line_width = 1
    label_suffix = f" ({current_fit_status.replace('Skipped_', '').replace('Fit_Failed_', 'Failed_')})"
  elif current_outlier_final_status == "OK":
    line_style = '-'
    alpha_val = 0.6
    line_width = 1.8
    label_suffix = " (OK)"
  elif current_outlier_final_status == "Outlier_Discarded":
      line_style = '--'
      alpha_val = 0.8
      color_val = 'red'
      marker_style = 'v'
      line_width = 1.5
      label_suffix = " (Discarded Outlier)"
  elif current_outlier_final_status == "O3_flagged":
      line_style = '-.'
      alpha_val = 0.7
      marker_style = '^'
      line_width = 1.5
    
  ax_main.plot(pressure_grid, y, label=f"{iso_id}{label_suffix}", 
        linestyle=line_style, color=color_val, alpha=alpha_val, marker=marker_style, markersize=3, linewidth=line_width)


ax_main.plot(pressure_grid, consensus_mean, 'k-', linewidth=2.5, label="Consensus Mean", alpha=0.9)
ax_main.fill_between(pressure_grid, consensus_ci_lower, consensus_ci_upper, color='gray', alpha=0.2, label="Consensus Mean  95% CI") # Label updated

ax_main.set_xscale('log') 
ax_main.set_xlim([0.01, 60]) 
ax_main.set_ylim(bottom=0)

ax_main.xaxis.set_major_locator(mticker.LogLocator(base=10.0, numticks=10))
ax_main.xaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs=(0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9), numticks=10))
ax_main.xaxis.set_major_formatter(mticker.FormatStrFormatter('%g'))


ax_main.set_xlabel("Pressure (bar) (log scale)")
ax_main.set_ylabel("Adsorption (mmol/g)")
ax_main.set_title(f"ZIF-8  {current_gas_name}: Adsorption Isotherms & Consensus (Grade: {reproducibility})")
ax_main.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1), ncol=1)
ax_main.grid(True, linestyle='--', alpha=0.7, which='both')


ax_inset = fig.add_axes([0.65, 0.2, 0.28, 0.28]) 

for iso_id in df.groupby("isotherm_json").groups.keys():
  y = interp_data[iso_id]
  
  current_outlier_final_status = outlier_flags.get(iso_id, "Not_Analyzed")
  current_fit_status = isotherm_fit_status.get(iso_id, "Unknown")

  line_style = '-'
  alpha_val = 0.8 
  color_val = iso_color_map.get(iso_id, 'black')
  marker_style = 'o'
  line_width = 1.5 

  if current_fit_status != "Fit_Success":
    line_style = ':'
    alpha_val = 0.3
    color_val = 'grey'
    marker_style = 'x'
    line_width = 1
  elif current_outlier_final_status == "OK":
    line_style = '-'
    alpha_val = 0.7
    line_width = 1.8
  elif current_outlier_final_status == "Outlier_Discarded":
      line_style = '--'
      alpha_val = 0.8
      color_val = 'red'
      marker_style = 'v'
      line_width = 1.5
  elif current_outlier_final_status == "O3_flagged":
      line_style = '-.'
      alpha_val = 0.7
      marker_style = '^'
      line_width = 1.5
    
  ax_inset.plot(pressure_grid, y, 
         linestyle=line_style, color=color_val, alpha=alpha_val, marker=marker_style, markersize=3, linewidth=line_width)

ax_inset.plot(pressure_grid, consensus_mean, 'k-', linewidth=1.5, alpha=0.9)
ax_inset.fill_between(pressure_grid, consensus_ci_lower, consensus_ci_upper, color='gray', alpha=0.2, label="Consensus Mean  95% CI") # Label updated

# Inset Y-axis limits (specific for current gas)
if current_gas_name == "Methane":
  ax_inset.set_ylim([0, 0.4]) # Adjusted Y-axis for Methane low pressure 
elif current_gas_name == "Carbon Dioxide": # Assuming this is for CO2, or adjust to actual variable name
  ax_inset.set_ylim([0, 1.0]) # Adjusted Y-axis for CO2 low pressure (default 1.0)
else: # Fallback or if not explicitly Methane/CO2
  ax_inset.set_ylim(bottom=0)

ax_inset.set_xlim([0.1, 1.0]) 
ax_inset.set_title("Low Pressure Region (Linear)", fontsize=8)
ax_inset.set_xlabel("Pressure (bar)", fontsize=8)
ax_inset.set_ylabel("Adsorption (mmol/g)", fontsize=8)
ax_inset.tick_params(axis='both', which='major', labelsize=7)

ax_inset.xaxis.set_major_locator(mticker.MultipleLocator(0.2))
ax_inset.xaxis.set_minor_locator(mticker.MultipleLocator(0.05))
ax_inset.grid(True, linestyle='--', alpha=0.7, which='both')


plt.tight_layout(rect=[0, 0, 0.98, 1]) 

plot_filename = os.path.join(OUTPUT_DIR, f"ZIF-8__{current_gas_name.replace(' ', '_')}_Reproducibility_Plot.png")
plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
plt.close()
print(f" Saved plot: {plot_filename}")

# =========================================================================
# FINAL SUMMARY
# =========================================================================
print(f"\n" + "="*60)
print(f"PARK REPRODUCIBILITY FRAMEWORK - COMPLETE")
print(f"="*60)
print(f"Dataset: ZIF-8 with {current_gas_name}")
print(f"Total isotherms analyzed (original input): {len(df.groupby('isotherm_json').groups.keys())}")
print(f"Total isotherms eligible for outlier detection (successful fit): {N_prime}")
print(f'Valid isotherms retained for consensus: {N_prime}')
print(f'Reproducibility grade: {reproducibility}')
print(f'Outlier detection method: Dynamic O1/O2 per pressure point')
print(f'Output directory: {OUTPUT_DIR}')
print('Park reproducibility framework complete.')


### 3.3 Reproducibility Screening - Carbon Dioxide

In [ ]:
# =========================================================================
# PARK REPRODUCIBILITY FRAMEWORK (FULL IMPLEMENTATION FOR CARBON DIOXIDE)
# =========================================================================
# This script performs the full Park et al. reproducibility analysis for
# ZIF-8 with Carbon Dioxide, including dynamic outlier detection, mean-based
# consensus, and plots results using matplotlib with specific axis scaling.

# === CONFIGURATION PARAMETERS ===
INPUT_CSV = "./zif-code/isotherm_dataset_by_zif_gas_after_manual_validation/ZIF-8/ZIF-8__CARBON DIOXIDE.csv" 
OUTPUT_DIR = "park_reproducibility_output_final/carbondioxide"  
current_gas_name = 'Carbon Dioxide' # Define current_gas_name for consistent output messages

# --- Common Pressure Points (Log-spaced for granularity) ---
log_min_p = np.log10(0.01) # log10(0.01) = -2
log_max_p = np.log10(60.0) # log10(60) approx = 1.778

N_POINTS = 50 # Total number of evaluation points for the log-spaced grid
COMMON_PRESSURE_POINTS = np.logspace(log_min_p, log_max_p, N_POINTS)

# --- Physical Qm Limits (GLOBAL, then used for specific gas) ---
MAX_PHYSICAL_QM_METHANE = 9.98 # mmol/g for CH4 on ZIF-8 (from literature/pore volume estimates)
MAX_PHYSICAL_QM_CO2 = 13.31     # mmol/g for CO2 on ZIF-8 (from literature/pore volume estimates)

# --- Determine MAX_PHYSICAL_QM for the current gas ---
MAX_PHYSICAL_QM_FOR_CURRENT_GAS = MAX_PHYSICAL_QM_CO2

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)





# === STEP 1: GROUP DATA BY INDIVIDUAL ISOTHERMS AND STORE PRESSURE RANGES ===
print("=== STEP 1: GROUPING ISOTHERMS AND RECORDING RANGES ===")
df = pd.read_csv(INPUT_CSV)
grouped = df.groupby("isotherm_json")
print(f"Found {len(grouped)} unique isotherms in the dataset")

isotherm_pressure_ranges = {}
isotherm_data_distribution = {}

for isotherm_name, group_data in grouped:
    pressures = group_data["pressure"].dropna()
    adsorptions = group_data["adsorption_mmol_per_g"].dropna()

    isotherm_data_distribution[isotherm_name] = {
        "num_points": len(group_data),
        "min_pressure": pressures.min() if not pressures.empty else np.nan,
        "max_pressure": pressures.max() if not pressures.empty else np.nan,
        "min_adsorption": adsorptions.min() if not adsorptions.empty else np.nan,
        "max_adsorption": adsorptions.max() if not adsorptions.empty else np.nan,
    }

    if not pressures.empty and len(pressures) >= 2:
        isotherm_pressure_ranges[isotherm_name] = (pressures.min(), pressures.max())
        print(f"\n--- Processing Isotherm: {isotherm_name} ---")
        print(f"  Original data points: {isotherm_data_distribution[isotherm_name]['num_points']}")
        print(f"  Original pressure range: [{isotherm_data_distribution[isotherm_name]['min_pressure']:.2f}, {isotherm_data_distribution[isotherm_name]['max_pressure']:.2f}] bar")
        print(f"  Original adsorption range: [{isotherm_data_distribution[isotherm_name]['min_adsorption']:.2f}, {isotherm_data_distribution[isotherm_name]['max_adsorption']:.2f}] mmol/g")
    else:
        print(f"\n--- Skipping Isotherm: {isotherm_name} ---")
        print(f"   Insufficient original data points ({len(pressures)}) to determine pressure range (min 2 required).")

# === STEP 2: CREATE GLOBAL INTERPOLATION GRID ===
print("\n=== STEP 2: CREATING GLOBAL INTERPOLATION GRID ===")
pressure_grid = COMMON_PRESSURE_POINTS
print(f"Created global interpolation grid with {N_POINTS} pressure points from {pressure_grid.min():.2f} to {pressure_grid.max():.2f} bar.")
print(f"Global pressure grid points: {np.array2string(pressure_grid, precision=2, separator=', ')}")

# === STEP 3: FIT TOTH MODEL AND EVALUATE ISOTHERMS (ENHANCED DIAGNOSTICS & CONDITIONAL Qm) ===
print("\n=== STEP 3: FITTING TOTH MODEL AND EVALUATING ISOTHERMS (ENHANCED DIAGNOSTICS & CONDITIONAL Qm) ===")
interp_data = {}
isotherm_fit_status = {}
fitted_params_summary = {} 

for isotherm_id, group in grouped:
    print(f"\n--- Toth Fitting for: {isotherm_id} ---")
    p_exp = group["pressure"].values
    q_exp = group["adsorption_mmol_per_g"].values

    valid_indices = ~np.isnan(p_exp) & ~np.isnan(q_exp)
    p_exp_clean = p_exp[valid_indices]
    q_exp_clean = q_exp[valid_indices]

    print(f"  Clean data points for fitting: {len(p_exp_clean)}")
    if len(p_exp_clean) > 0:
        print(f"    Sample clean pressures (first 5): {np.array2string(p_exp_clean[:5], precision=4, separator=', ')}")
        print(f"    Sample clean adsorptions (first 5): {np.array2string(q_exp_clean[:5], precision=4, separator=', ')}")
        if len(p_exp_clean) > 5:
            print(f"    Sample clean pressures (last 5): {np.array2string(p_exp_clean[-5:], precision=4, separator=', ')}")
            print(f"    Sample clean adsorptions (last 5): {np.array2string(q_exp_clean[-5:], precision=4, separator=', ')}")

    if isotherm_id not in isotherm_pressure_ranges:
        print(f"   Skipping Toth fit: No valid pressure range recorded in Step 1.")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Skipped_NoOriginalRange"
        continue

    if len(p_exp_clean) < 3:
        print(f"   Skipping Toth fit: Insufficient clean data points ({len(p_exp_clean)}) for Toth model fitting (min 3 required).")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Skipped_InsufficientData"
        continue
    
    # Heuristic for deciding whether to fix Qm
    if np.max(q_exp_clean) < (MAX_PHYSICAL_QM_FOR_CURRENT_GAS / 2.0):
        use_fixed_Qm = True
        fixed_Qm_val_for_fit = MAX_PHYSICAL_QM_FOR_CURRENT_GAS
        print(f"  Deciding to FIX Qm: Max adsorption ({np.max(q_exp_clean):.2f}) is low, fitting with fixed Qm={fixed_Qm_val_for_fit:.2f}.")
    else:
        use_fixed_Qm = False
        print(f"  Deciding to FIT Qm: Max adsorption ({np.max(q_exp_clean):.2f}) is high enough.")


    try:
        if use_fixed_Qm:
            initial_guesses = [1.0, 0.9] # K_t, t
            # bounds = ([1e-6, 1e-6], [np.inf, 2.0]) # K_t bounds, t bounds (t up to 2.0)
            bounds = ([1e-6, 1e-6], [np.inf, 0.98]) # K_t bounds, t bounds (t up to 2.0)
            
            params, covariance = curve_fit(lambda p, K_t, t: toth_model_fixed_Qm(p, K_t, t, fixed_Qm_val_for_fit),
                                           p_exp_clean, q_exp_clean,
                                           p0=initial_guesses, bounds=bounds, maxfev=20000, 
                                           ftol=1e-8, xtol=1e-8, gtol=1e-8)
            K_t_fit, t_fit = params
            Q_m_fit = fixed_Qm_val_for_fit # Record the fixed Qm
            print(f"   Toth Fit (Qm FIXED) Successful!")
            print(f"    Fitted Parameters: Q_m={Q_m_fit:.4f} (FIXED), K_t={K_t_fit:.4f}, t={t_fit:.4f}")
            fitted_params_summary[isotherm_id] = {'Q_m': Q_m_fit, 'K_t': K_t_fit, 't': t_fit, 'fit_type': 'Fixed_Qm'}

        else:
            initial_guesses = [np.max(q_exp_clean) * 1.2, 1.0, 1.0]
            if initial_guesses[0] <= 0 or np.isnan(initial_guesses[0]):
                initial_guesses[0] = 0.1
            # bounds = ([1e-6, 1e-6, 1e-6], [np.inf, np.inf, 2.0])
            bounds = ([1e-6, 1e-6, 1e-6], [np.inf, np.inf, 0.99])
            
            params, covariance = curve_fit(toth_model_full, p_exp_clean, q_exp_clean,
                                           p0=initial_guesses, bounds=bounds, maxfev=20000, 
                                           ftol=1e-8, xtol=1e-8, gtol=1e-8)
            Q_m_fit, K_t_fit, t_fit = params
            print(f"   Toth Fit (Full) Successful!")
            print(f"    Fitted Parameters: Q_m={Q_m_fit:.4f}, K_t={K_t_fit:.4f}, t={t_fit:.4f}")
            fitted_params_summary[isotherm_id] = {'Q_m': Q_m_fit, 'K_t': K_t_fit, 't': t_fit, 'fit_type': 'Full_Fit'}

        # Check if parameters hit bounds 
        kt_lower_bound_val = bounds[0][0] if use_fixed_Qm else bounds[0][1]
        kt_upper_bound_val = bounds[1][0] if use_fixed_Qm else bounds[1][1]
        t_lower_bound_val = bounds[0][1] if use_fixed_Qm else bounds[0][2]
        t_upper_bound_val = bounds[1][1] if use_fixed_Qm else bounds[1][2]

        if np.isclose(K_t_fit, kt_lower_bound_val) or np.isclose(K_t_fit, kt_upper_bound_val):
            print(f"     Warning: K_t ({K_t_fit:.4f}) hit a bound.")
        
        if np.isclose(t_fit, t_lower_bound_val) or np.isclose(t_fit, t_upper_bound_val):
            print(f"     Warning: t ({t_fit:.4f}) hit a bound.")
        
        if not use_fixed_Qm and (np.isclose(Q_m_fit, bounds[0][0]) or np.isclose(Q_m_fit, bounds[1][0])):
             print(f"     Warning: Q_m ({Q_m_fit:.4f}) hit a bound.")
        if not use_fixed_Qm and Q_m_fit > MAX_PHYSICAL_QM_FOR_CURRENT_GAS * 1.5: 
            print(f"     Warning: Fitted Qm ({Q_m_fit:.2f}) is significantly above physical limit ({MAX_PHYSICAL_QM_FOR_CURRENT_GAS:.2f}).")


        interp_q = toth_model_full(pressure_grid, Q_m_fit, K_t_fit, t_fit)

        min_original_p_val, max_original_p_val = isotherm_pressure_ranges[isotherm_id]
        mask_above_max = (pressure_grid > max_original_p_val)
        
        interp_q[mask_above_max] = np.nan
        
        interp_data[isotherm_id] = interp_q
        isotherm_fit_status[isotherm_id] = "Fit_Success"
        print(f"   Evaluated on global grid. Values ABOVE original range masked with NaN (allowing extrapolation below min).")
        sample_indices = [0, len(pressure_grid)//4, len(pressure_grid)//2, 3*len(pressure_grid)//4, len(pressure_grid)-1]
        sample_pressures = pressure_grid[sample_indices]
        sample_adsorptions = interp_q[sample_indices]
        print(f"    Sample evaluated P (bar): {np.array2string(sample_pressures, precision=2, separator=', ')}")
        print(f"    Sample evaluated Q (mmol/g): {np.array2string(sample_adsorptions, precision=4, separator=', ')}")


    except RuntimeError as e:
        print(f"   Toth Fit FAILED: {e}. (Could not converge or find optimal parameters)")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Fit_Failed_Runtime"
    except ValueError as e:
        print(f"   Toth Fit FAILED: {e}. (Invalid parameters or data encountered, e.g., all clean_q_exp is zero or negative)")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Fit_Failed_Value"
    except Exception as e:
        print(f"   Toth Fit FAILED: An unexpected error occurred: {e}")
        interp_data[isotherm_id] = np.full(N_POINTS, np.nan)
        isotherm_fit_status[isotherm_id] = "Fit_Failed_Other"

print(f"\n=== Toth Model Fitting Stage Complete ===")
print(f"Status for all isotherms:")
for iso_id, status in isotherm_fit_status.items():
  print(f" - {iso_id}: {status}")

# --- Display DataFrame of all Toth-fitted isotherms ---
print(f"\n=== Toth Fitted Isotherms Data (mmol/g) ===")
df_display_data = {
  iso_id: interp_data[iso_id] for iso_id in sorted(interp_data.keys())
}
df_toth_fitted = pd.DataFrame(df_display_data, index=pressure_grid)
df_toth_fitted.index.name = "Pressure (bar)"
print(df_toth_fitted.to_string(float_format="%.4f"))


# =========================================================================
# STEP 4: OUTLIER DETECTION (FULL IMPLEMENTATION AS PER SUPERVISOR DIRECTIVE)
# =========================================================================
print(f"\n=== STEP 4: OUTLIER DETECTION ===")

eligible_isotherm_ids = [iso_id for iso_id, status in isotherm_fit_status.items() if status == "Fit_Success"]
eligible_matrix = np.array([interp_data[iso_id] for iso_id in eligible_isotherm_ids])
N_eligible = len(eligible_isotherm_ids)

print(f"Analyzing {N_eligible} eligible isotherms for outliers...")

outlier_flags = {iso_id: "Not_Analyzed" for iso_id in df.groupby("isotherm_json").groups.keys()}
isotherm_flagged_points_count = {iso_id: 0 for iso_id in eligible_isotherm_ids} # Reset for a clean count this run
isotherm_rmse_values = {iso_id: np.nan for iso_id in eligible_isotherm_ids} # For O2 method only

print("\n Outlier detection results per pressure point:")
for j, p_val in enumerate(pressure_grid):
  values_at_pressure = eligible_matrix[:, j]
  values_at_pressure_clean = values_at_pressure[~np.isnan(values_at_pressure)]
  
  N_local = len(values_at_pressure_clean)
  
  # Collect flags for this pressure point
  flags_at_this_point = [] 

  if N_local < 2:
    print(f"  P={p_val:.2f} bar: {N_local} isotherms contribute. Skipping outlier check (N_local < 2).")
    continue 

  # Determine method based on N_local
  if N_local > 4: # O1 Method (Tukey's Rule)
    q1 = np.percentile(values_at_pressure_clean, 25)
    q3 = np.percentile(values_at_pressure_clean, 75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    for i_iso, iso_id in enumerate(eligible_isotherm_ids):
      current_value = eligible_matrix[i_iso, j]
      if not np.isnan(current_value):
        if current_value < lower_bound or current_value > upper_bound:
          flags_at_this_point.append(f" {iso_id} (Val={current_value:.2f}, Bounds=[{lower_bound:.2f}, {upper_bound:.2f}], Method: O1)")
          isotherm_flagged_points_count[iso_id] += 1
  
  elif N_local in [3, 4]: # O2 Method (RMSE/sigma comparison)
    median_uptake = np.nanmedian(values_at_pressure_clean)
    std_uptake = np.nanstd(values_at_pressure_clean)
    
    if np.isnan(std_uptake) or std_uptake == 0:
      print(f"    Cannot apply O2 method at P={p_val:.2f} due to zero/NaN std_uptake.")
      continue

    potential_flags_at_this_point = []
    for i_iso, iso_id in enumerate(eligible_isotherm_ids):
      current_value = eligible_matrix[i_iso, j]
      if not np.isnan(current_value):
        rmse_point = np.sqrt(mean_squared_error([median_uptake], [current_value]))
        
        if rmse_point > std_uptake / 2:
          potential_flags_at_this_point.append(iso_id)
    
    # --- NEW ROBUSTNESS CHECK FOR N_LOCAL = 3 or 4 ---
    if len(potential_flags_at_this_point) > 1: # If 2 or more would be flagged
      print(f"    P={p_val:.2f}: N_local={N_local}. {len(potential_flags_at_this_point)} isotherms potentially flagged. (High spread, no distinct outliers at this point).")
    else: # If 0 or 1 isotherm is flagged, then apply the flag.
      for iso_id_flagged in potential_flags_at_this_point:
        isotherm_flagged_points_count[iso_id_flagged] += 1
        print(f"    {iso_id_flagged} FLAGGED at P={p_val:.2f} (RMSE={rmse_point:.2f}, Std/2={std_uptake/2:.2f}). Method: O2")
  # Print summary for this pressure point
  if flags_at_this_point:
    print(f"  P={p_val:.2f} bar: {N_local} isotherms contribute. Flags: {' '.join(flags_at_this_point)}")
  else:
    print(f"  P={p_val:.2f} bar: {N_local} isotherms contribute. No outliers found at this point.")

print("\n Applying supervisor's outlier discard rule:")
for iso_id in eligible_isotherm_ids:
  flagged_count = isotherm_flagged_points_count[iso_id]
  
  if flagged_count <= 4:
    outlier_flags[iso_id] = "OK"
    print(f"   {iso_id}: Flagged at {flagged_count} points. KEPT (OK).")
  else:
    outlier_flags[iso_id] = "Outlier_Discarded"
    print(f"   {iso_id}: Flagged at {flagged_count} points. DISCARDED (Outlier).")

N_prime = sum(1 for flag in outlier_flags.values() if flag == "OK")
outlier_count_final = sum(1 for flag in outlier_flags.values() if flag == "Outlier_Discarded")
print(f"Total {N_prime} isotherms are OK for consensus.")
print(f"Total {outlier_count_final} isotherms are discarded as outliers.")


# === STEP 5: ASSIGN REPRODUCIBILITY GRADE ===
print(f"\n=== STEP 5: ASSIGN REPRODUCIBILITY GRADING ===")

if N_eligible > 2:
  f = (N_eligible - N_prime) / N_eligible
  print(f"Outlier fraction (f): {f:.2f}")
else:
  f = None
  print(f"Outlier fraction (f): Not applicable due to N_eligible < 3.")


if N_eligible > 4:
  if f is not None and f <= 0.25:
    reproducibility = "R1"
    grade_explanation = "Excellent: 25% outliers"
  elif f is not None and f <= 0.4:
    reproducibility = "R2"
    grade_explanation = "Good: 25-40% outliers"
  else:
    reproducibility = "R3"
    grade_explanation = "Moderate: >40% outliers"

elif N_eligible in [3, 4]:
  reproducibility = "R2"
  grade_explanation = "Good: 3-4 eligible isotherms available"

elif N_eligible == 2:
  curve1 = eligible_matrix[0]
  curve2 = eligible_matrix[1]
  common_valid_points = ~np.isnan(curve1) & ~np.isnan(curve2)
  
  if np.sum(common_valid_points) < 1:
    reproducibility = "Insufficient data"
    grade_explanation = "Cannot assess with unreliable RMSE/sigma for 2 isotherms (no common points)"
  else:
    rmse_between_curves = np.sqrt(mean_squared_error(curve1[common_valid_points], curve2[common_valid_points]))
    combined_matrix_for_std = np.array([curve1[common_valid_points], curve2[common_valid_points]])
    sigma_overall = np.mean(np.std(combined_matrix_for_std, axis=0))
    
    if rmse_between_curves <= sigma_overall / 2:
      reproducibility = "R3"
      grade_explanation = "Moderate: Low spread between 2 isotherms"
    else:
      reproducibility = "R4"
      grade_explanation = "Poor: High spread between 2 isotherms"

else: # N_eligible < 2
  reproducibility = "Insufficient data"
  grade_explanation = "Cannot assess with <2 eligible isotherms"

print(f"Reproducibility Grade: {reproducibility}")
print(f"Explanation: {grade_explanation}")


# =========================================================================
# STEP 6: CREATE CONSENSUS ISOTHERM (Mean uptake & Standard Deviation)
# =========================================================================
print(f"\n=== STEP 6: CONSENSUS ISOTHERM GENERATION ===")

valid_curves_for_consensus = []
for iso_id in eligible_isotherm_ids:
  if outlier_flags.get(iso_id) == "OK":
    valid_curves_for_consensus.append(interp_data[iso_id])
    print(f"Including {iso_id} in consensus (OK status).")
  else:
    print(f"Excluding {iso_id} from consensus (flagged: {outlier_flags.get(iso_id, 'Not OK')}).")

if valid_curves_for_consensus:
  # Calculate initial consensus mean and std from OK isotherms only
  consensus_mean = np.nanmean(valid_curves_for_consensus, axis=0)
  consensus_std = np.nanstd(valid_curves_for_consensus, axis=0)
  print(f"Consensus based on {len(valid_curves_for_consensus)} OK isotherms.")
else:
  # Fallback for if NO isotherms were OK AT ALL
  print(" No isotherms marked as 'OK' for consensus. Calculating consensus (mean & std) from ALL eligible isotherms as fallback.")
  consensus_mean = np.nanmean(eligible_matrix, axis=0)
  consensus_std = np.nanstd(eligible_matrix, axis=0)
  print(f"Consensus based on all {N_eligible} eligible isotherms (no 'OK' exclusions).")

# --- NEW: Hybrid Consensus Logic (Fill NaNs with all eligible data) ---
# This loop will fill NaNs in consensus_mean and consensus_std if they occurred due to sparsity
# even when 'OK' isotherms existed at other pressures.
# It will use *all eligible isotherms* for those specific points where primary consensus is NaN.
consensus_mean_hybrid = consensus_mean.copy()
consensus_std_hybrid = consensus_std.copy()
N_local_consensus = np.full(N_POINTS, np.nan) # To store N_local for CI calculation

for j in range(N_POINTS):
  current_values_ok = np.array([curve[j] for curve in valid_curves_for_consensus if not np.isnan(curve[j])])
  
  if len(current_values_ok) > 0:
    N_local_consensus[j] = len(current_values_ok)
  else: # If primary consensus is NaN at this point, use all eligible data
    values_at_pressure_all_eligible = eligible_matrix[:, j]
    values_at_pressure_all_eligible_clean = values_at_pressure_all_eligible[~np.isnan(values_at_pressure_all_eligible)]

    if len(values_at_pressure_all_eligible_clean) > 0:
      consensus_mean_hybrid[j] = np.nanmean(values_at_pressure_all_eligible_clean)
      consensus_std_hybrid[j] = np.nanstd(values_at_pressure_all_eligible_clean)
      N_local_consensus[j] = len(values_at_pressure_all_eligible_clean)
      print(f" P={pressure_grid[j]:.2f} bar: Consensus was NaN. Filled using mean/std of {len(values_at_pressure_all_eligible_clean)} eligible isotherms (including discarded).")
    else:
      N_local_consensus[j] = 0 # No data even from all eligible

# Reassign consensus variables to the hybrid version for final output
consensus_mean = consensus_mean_hybrid
consensus_std = consensus_std_hybrid

# --- Calculate 95% Confidence Interval ---
confidence_level = 0.95
alpha = 1.0 - confidence_level

consensus_sem = np.full(N_POINTS, np.nan)
consensus_ci_lower = np.full(N_POINTS, np.nan)
consensus_ci_upper = np.full(N_POINTS, np.nan)

for j in range(N_POINTS):
  n = N_local_consensus[j] # Use the N_local that contributed to the final consensus_mean at this point
  current_std = consensus_std[j] # Use the hybrid std for CI calculation
  current_mean = consensus_mean[j] # Use the hybrid mean

  if n > 1 and not np.isnan(current_std): # Need at least 2 points for std dev, and n-1 degrees of freedom for t-dist
    sem = current_std / np.sqrt(n)
    degrees_freedom = n - 1
    t_score = t_dist.ppf(1 - alpha / 2, degrees_freedom) # Two-tailed t-score
    
    consensus_sem[j] = sem
    consensus_ci_lower[j] = current_mean - t_score * sem
    consensus_ci_upper[j] = current_mean + t_score * sem
  elif n == 1 and not np.isnan(current_std): # If only one point, SD is 0, SEM is 0. CI is just the point itself.
    consensus_sem[j] = 0.0 # SEM is effectively 0 for N=1
    consensus_ci_lower[j] = current_mean
    consensus_ci_upper[j] = current_mean
  # If n=0 or nan, CI remains nan.

# If all values in consensus_mean are NaN (e.g., no data points at all)
if np.all(np.isnan(consensus_mean)):
  print("Warning: Consensus mean is all NaNs. This might indicate no data points at any pressure.")
  consensus_mean = np.zeros(N_POINTS)
  consensus_std = np.zeros(N_POINTS)
  consensus_sem = np.zeros(N_POINTS)
  consensus_ci_lower = np.zeros(N_POINTS)
  consensus_ci_upper = np.zeros(N_POINTS)

# Adjust consensus_plus_sd / consensus_minus_sd to represent CI
consensus_plus_sd = consensus_ci_upper # Now represents upper CI bound
consensus_minus_sd = consensus_ci_lower # Now represents lower CI bound


# =========================================================================
# STEP 7: SAVE OUTPUTS & PLOT (MATPLOTLIB)
# =========================================================================
print(f"\n=== STEP 7: SAVING RESULTS & PLOTTING ===")

# === SAVE CONSENSUS ISOTHERM CSV ===
consensus_df = pd.DataFrame({
  "pressure_bar": pressure_grid,
  "consensus_adsorption_mean_mmol_g": consensus_mean,
  "consensus_adsorption_std_mmol_g": consensus_std,
  "consensus_adsorption_sem_mmol_g": consensus_sem, # NEW
  "consensus_adsorption_ci_lower_mmol_g": consensus_ci_lower, # NEW
  "consensus_adsorption_ci_upper_mmol_g": consensus_ci_upper, # NEW
})
consensus_filename = os.path.join(OUTPUT_DIR, f"ZIF-8__{current_gas_name.replace(' ', '_')}_consensus_isotherm.csv") # Fixed filename for saving
consensus_df.to_csv(consensus_filename, index=False)
print(f" Saved consensus isotherm (mean and std): {consensus_filename}")

# --- NEW: Save All Individual Isotherms + Consensus to One CSV ---
combined_output_df = df_toth_fitted.copy() # Start with the Toth-fitted individual isotherms
combined_output_df['Consensus_Mean_mmol_g'] = consensus_mean
combined_output_df['Consensus_Std_mmol_g'] = consensus_std
combined_output_df['Consensus_SEM_mmol_g'] = consensus_sem # NEW
combined_output_df['Consensus_CI_Lower_mmol_g'] = consensus_ci_lower # NEW
combined_output_df['Consensus_CI_Upper_mmol_g'] = consensus_ci_upper # NEW

combined_output_filename = os.path.join(OUTPUT_DIR, f"ZIF-8__{current_gas_name.replace(' ', '_')}_all_isotherms_and_consensus.csv") # Fixed filename for saving
combined_output_df.to_csv(combined_output_filename, index=True) # index=True to include Pressure (bar)
print(f" Saved all Toth-fitted isotherms and consensus to: {combined_output_filename}")


summary_rows = []
for iso_id in df.groupby("isotherm_json").groups.keys(): # Iterate over all original isotherms
  summary_rows.append({
    "isotherm_json": iso_id,
    "fit_status": isotherm_fit_status.get(iso_id, "Unknown"),
    "outlier_final_status": outlier_flags.get(iso_id, "Not_Analyzed"),
    "flagged_points_count": isotherm_flagged_points_count.get(iso_id, np.nan) if isotherm_fit_status.get(iso_id) == "Fit_Success" else np.nan,
    "reproducibility_grade": reproducibility # Global grade applies to all in summary
  })

summary_df = pd.DataFrame(summary_rows)
summary_filename = os.path.join(OUTPUT_DIR, f"ZIF-8__{current_gas_name.replace(' ', '_')}_isotherm_summary.csv") # Fixed filename for saving
summary_df.to_csv(summary_filename, index=False)
print(f" Saved summary table: {summary_filename}")


# === CREATE VISUALIZATION (MATPLOTLIB) ===
print("Creating visualization (matplotlib)...")
fig, ax_main = plt.subplots(figsize=(10, 6))

cmap = plt.cm.get_cmap('tab10', len(df.groupby("isotherm_json").groups.keys()))
colors_list = [cmap(i) for i in range(len(df.groupby("isotherm_json").groups.keys()))]
iso_color_map = {iso_id: colors_list[i] for i, iso_id in enumerate(df.groupby("isotherm_json").groups.keys())}


for iso_id in df.groupby("isotherm_json").groups.keys():
  y = interp_data[iso_id]
  
  current_outlier_final_status = outlier_flags.get(iso_id, "Not_Analyzed")
  current_fit_status = isotherm_fit_status.get(iso_id, "Unknown")

  label_suffix = ""
  line_style = '-'
  alpha_val = 0.8
  color_val = iso_color_map.get(iso_id, 'black')
  marker_style = 'o'
  line_width = 1.5

  if current_fit_status != "Fit_Success":
    line_style = ':'
    alpha_val = 0.3
    color_val = 'grey'
    marker_style = 'x'
    line_width = 1
    label_suffix = f" ({current_fit_status.replace('Skipped_', '').replace('Fit_Failed_', 'Failed_')})"
  elif current_outlier_final_status == "OK":
    line_style = '-'
    alpha_val = 0.6
    line_width = 1.8
    label_suffix = " (OK)"
  elif current_outlier_final_status == "Outlier_Discarded":
      line_style = '--'
      alpha_val = 0.8
      color_val = 'red'
      marker_style = 'v'
      line_width = 1.5
      label_suffix = " (Discarded Outlier)"
  elif current_outlier_final_status == "O3_flagged":
      line_style = '-.'
      alpha_val = 0.7
      marker_style = '^'
      line_width = 1.5
    
  ax_main.plot(pressure_grid, y, label=f"{iso_id}{label_suffix}", 
        linestyle=line_style, color=color_val, alpha=alpha_val, marker=marker_style, markersize=3, linewidth=line_width)


ax_main.plot(pressure_grid, consensus_mean, 'k-', linewidth=2.5, label="Consensus Mean", alpha=0.9)
ax_main.fill_between(pressure_grid, consensus_ci_lower, consensus_ci_upper, color='gray', alpha=0.2, label="Consensus Mean  95% CI") # Label updated

ax_main.set_xscale('log') 
ax_main.set_xlim([0.01, 60]) 
ax_main.set_ylim(bottom=0)

ax_main.xaxis.set_major_locator(mticker.LogLocator(base=10.0, numticks=10))
ax_main.xaxis.set_minor_locator(mticker.LogLocator(base=10.0, subs=(0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9), numticks=10))
ax_main.xaxis.set_major_formatter(mticker.FormatStrFormatter('%g'))


ax_main.set_xlabel("Pressure (bar) (log scale)")
ax_main.set_ylabel("Adsorption (mmol/g)")
ax_main.set_title(f"ZIF-8  {current_gas_name}: Adsorption Isotherms & Consensus (Grade: {reproducibility})")
ax_main.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1), ncol=1)
ax_main.grid(True, linestyle='--', alpha=0.7, which='both')


ax_inset = fig.add_axes([0.65, 0.2, 0.28, 0.28]) 

for iso_id in df.groupby("isotherm_json").groups.keys():
  y = interp_data[iso_id]
  
  current_outlier_final_status = outlier_flags.get(iso_id, "Not_Analyzed")
  current_fit_status = isotherm_fit_status.get(iso_id, "Unknown")

  line_style = '-'
  alpha_val = 0.8 
  color_val = iso_color_map.get(iso_id, 'black')
  marker_style = 'o'
  line_width = 1.5 

  if current_fit_status != "Fit_Success":
    line_style = ':'
    alpha_val = 0.3
    color_val = 'grey'
    marker_style = 'x'
    line_width = 1
  elif current_outlier_final_status == "OK":
    line_style = '-'
    alpha_val = 0.7
    line_width = 1.8
  elif current_outlier_final_status == "Outlier_Discarded":
      line_style = '--'
      alpha_val = 0.8
      color_val = 'red'
      marker_style = 'v'
      line_width = 1.5
  elif current_outlier_final_status == "O3_flagged":
      line_style = '-.'
      alpha_val = 0.7
      marker_style = '^'
      line_width = 1.5
    
  ax_inset.plot(pressure_grid, y, 
         linestyle=line_style, color=color_val, alpha=alpha_val, marker=marker_style, markersize=3, linewidth=line_width)

ax_inset.plot(pressure_grid, consensus_mean, 'k-', linewidth=1.5, alpha=0.9)
ax_inset.fill_between(pressure_grid, consensus_ci_lower, consensus_ci_upper, color='gray', alpha=0.2, label="Consensus Mean  95% CI") # Label updated

# Inset Y-axis limits (specific for current gas)
if current_gas_name == "Methane":
  ax_inset.set_ylim([0, 0.3]) # Adjusted Y-axis for Methane low pressure 
elif current_gas_name == "Carbon Dioxide": # Assuming this is for CO2, or adjust to actual variable name
  ax_inset.set_ylim([0, 1.0]) # Adjusted Y-axis for CO2 low pressure (default 1.0)
else: # Fallback or if not explicitly Methane/CO2
  ax_inset.set_ylim(bottom=0)

ax_inset.set_xlim([0.1, 1.0]) 
ax_inset.set_title("Low Pressure Region (Linear)", fontsize=8)
ax_inset.set_xlabel("Pressure (bar)", fontsize=8)
ax_inset.set_ylabel("Adsorption (mmol/g)", fontsize=8)
ax_inset.tick_params(axis='both', which='major', labelsize=7)

ax_inset.xaxis.set_major_locator(mticker.MultipleLocator(0.2))
ax_inset.xaxis.set_minor_locator(mticker.MultipleLocator(0.05))
ax_inset.grid(True, linestyle='--', alpha=0.7, which='both')


plt.tight_layout(rect=[0, 0, 0.98, 1]) 

plot_filename = os.path.join(OUTPUT_DIR, f"ZIF-8__{current_gas_name.replace(' ', '_')}_Reproducibility_Plot.png")
plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
plt.close()
print(f" Saved plot: {plot_filename}")

# =========================================================================
# FINAL SUMMARY
# =========================================================================
print(f"\n" + "="*60)
print(f"PARK REPRODUCIBILITY FRAMEWORK - COMPLETE")
print(f"="*60)
print(f"Dataset: ZIF-8 with {current_gas_name}")
print(f"Total isotherms analyzed (original input): {len(df.groupby('isotherm_json').groups.keys())}")
print(f"Total isotherms eligible for outlier detection (successful fit): {N_prime}")
print(f'Valid isotherms retained for consensus: {N_prime}')
print(f'Reproducibility grade: {reproducibility}')
print(f'Outlier detection method: Dynamic O1/O2 per pressure point')
print(f'Output directory: {OUTPUT_DIR}')
print('Park reproducibility framework complete.')


## Section 4: Consensus Isotherm Construction

After reproducibility screening produces per-gas consensus CSVs, this section reads the consolidated consensus Excel file and generates comparison plots showing individual Toth-fitted isotherms alongside the mean consensus and 95% confidence interval bands. Discarded outlier isotherms are highlighted in a distinct style to make the screening outcome visually clear.

In [ ]:
# === Config ===
filename = "./5_co2methaneconsensus.xlsx"
pressure_threshold = 17.3147153990574
output_dir = "./plots"
os.makedirs(output_dir, exist_ok=True)

# === Discarded Isotherms Mapping ===
discarded_map = {
 'co2': ["10.1021acs.jpcc.5b01519.Isotherm18.json"],
 'ch4': ["10.1016j.seppur.2019.115850.Isotherm8.json"]
}

# === Plot Function ===
def plot_isotherms(df, pressure, consensus_mean, ci_lower, ci_upper, isotherm_cols,
 gas_label, zoom=False):

 fig, ax = plt.subplots(figsize=(10, 6))

 error_lower = consensus_mean - ci_lower
 error_upper = ci_upper - consensus_mean
 mask_error = pressure <= pressure_threshold

 # Use distinct colors from matplotlib colormap
 cmap = plt.get_cmap('tab20')
 color_cycle = [cmap(i) for i in range(len(isotherm_cols))]

 for idx, col in enumerate(isotherm_cols):
 is_discarded = col in discarded_map.get(gas_label, [])
 label = col.replace('.json', '')
 linestyle = ':' if is_discarded else '--' # dotted if discarded
 if is_discarded:
 label = f"DISCARDED – {label}"

 ax.plot(
 pressure, df[col],
 linestyle=linestyle,
 linewidth=1.2,
 alpha=0.9,
 label=label,
 color=color_cycle[idx % len(color_cycle)]
 )

 # Consensus line
 ax.plot(pressure, consensus_mean, 'o-', color='black', linewidth=1.5, markersize=4,
 label='Consensus Mean', zorder=5)

 # Consensus CI error bars
 ax.errorbar(
 pressure[mask_error], consensus_mean[mask_error],
 yerr=[error_lower[mask_error], error_upper[mask_error]],
 fmt='none', ecolor='gray', elinewidth=1, capsize=3,
 label=f'Consensus CI (≤ {pressure_threshold:.2f} bar)', zorder=4
 )

 ax.set_xlabel("Pressure [bar]")
 ax.set_ylabel("Uptake [mmol/g]")
 ax.grid(True, which='both', linestyle='--', linewidth=0.5)
 ax.legend(loc="best", fontsize='x-small')

 if zoom:
 ax.set_xlim(0.01, 1.05)
 yaxis_max = 0.35 if gas_label == 'ch4' else 0.85
 ax.set_ylim(0.01, yaxis_max)
 ax.set_xscale('linear')
 ax.set_yticks(np.linspace(0.01, yaxis_max, 6))
 # ax.set_title(f"{gas_label.upper()} Isotherms (Zoomed: 0.01–1 bar)")
 filename_out = f"{gas_label}_zoom.png"
 else:
 ax.set_xscale('log')
 # ax.set_title(f"{gas_label.upper()} Isotherms (Full Range)")
 filename_out = f"{gas_label}_full.png"

 plt.tight_layout()
 plt.savefig(os.path.join(output_dir, filename_out), dpi=300)
 plt.show()
 plt.close()
 print(f"Saved: {filename_out}")

# === Main Execution ===
for gas_label in ['co2', 'ch4']:
 df = pd.read_excel(filename, sheet_name=gas_label)
 df = df.dropna(subset=["Pressure (bar)"])

 pressure = df["Pressure (bar)"]
 consensus_mean = df["Consensus_Mean_mmol_g"]
 ci_lower = df["Consensus_CI_Lower_mmol_g"]
 ci_upper = df["Consensus_CI_Upper_mmol_g"]
 isotherm_cols = [col for col in df.columns if col.endswith('.json')]

 plot_isotherms(df, pressure, consensus_mean, ci_lower, ci_upper, isotherm_cols, gas_label, zoom=False)
 plot_isotherms(df, pressure, consensus_mean, ci_lower, ci_upper, isotherm_cols, gas_label, zoom=True)


## Section 5: Isotherm Model Fitting

This section fits the Toth isotherm model to the consensus pure-component loading data for CO2 and CH4 on ZIF-8. The embedded consensus data (from the reproducibility screening outputs) is used as the fitting target. Fitted parameters and goodness-of-fit metrics (RMSE, R-squared) are reported. These parameters are subsequently used as inputs to the IAST mixture prediction sweep.

In [ ]:
import ruptura
from matplotlib.colors import LogNorm
# --- 1. CONFIGURATION PARAMETERS ---
OUTPUT_DIR = "ruptura_iast_results_sweep"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f" Created output directory: {OUTPUT_DIR}")

# --- Parameters for the Sweep Calculation ---
FEED_CO2_FRACTIONS = [
    0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.25, 
    0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 0.75, 0.8, 0.9, 0.91, 0.92, 
    0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99
]
PRESSURE_SWEEP_BAR = [
     0.01, 0.011942772, 0.014262979, 0.017033951, 0.020343258, 0.024295489, 0.029015547, 0.034652605,
    0.041384815, 0.04942494, 0.059027077, 0.07049469, 0.084190198, 0.100546431, 0.120080306,
    0.143409167, 0.171270293, 0.2045442, 0.244282466, 0.291740971, 0.348419579, 0.416109546,
    0.496950127, 0.593496188, 0.708798943, 0.846502391, 1.010958473, 1.207364616, 1.441927988,
    1.722061666, 2.05661892, 2.456173008, 2.93335133, 3.503234502, 4.18383296, 4.996656156,
    5.967392335, 7.126720385, 8.511279398, 10.16482661, 12.13962028, 14.49807126, 17.3147154,
    20.67856917, 24.69594291, 29.49380063, 35.22377252, 42.06694709, 50.2395942
]

PRESSURE_SWEEP_PA = [p * 1e5 for p in PRESSURE_SWEEP_BAR]

# --- 2. EMBEDDED CONSENSUS DATA ---
print(" Loading consensus data...")
pressure_bar_data = np.array([
    0.01, 0.011942772, 0.014262979, 0.017033951, 0.020343258, 0.024295489, 0.029015547, 0.034652605,
    0.041384815, 0.04942494, 0.059027077, 0.07049469, 0.084190198, 0.100546431, 0.120080306,
    0.143409167, 0.171270293, 0.2045442, 0.244282466, 0.291740971, 0.348419579, 0.416109546,
    0.496950127, 0.593496188, 0.708798943, 0.846502391, 1.010958473, 1.207364616, 1.441927988,
    1.722061666, 2.05661892, 2.456173008, 2.93335133, 3.503234502, 4.18383296, 4.996656156,
    5.967392335, 7.126720385, 8.511279398, 10.16482661, 12.13962028, 14.49807126, 17.3147154,
    20.67856917, 24.69594291, 29.49380063, 35.22377252, 42.06694709, 50.2395942
])
pressure_pascal_data = pressure_bar_data * 1e5
co2_loading_data = np.array([
    0.007168033, 0.008553413, 0.010205744, 0.012176217, 0.014525756, 0.017326852, 0.020665719,
    0.024644849, 0.029386027, 0.035033878, 0.041760046, 0.049768096, 0.059299255, 0.070639117,
    0.084125444, 0.10015721, 0.119205032, 0.141823132, 0.168662951, 0.200488498, 0.238193456,
    0.282819942, 0.33557865, 0.397869808, 0.471304004, 0.557721253, 0.69753242, 0.868904942,
    1.03531449, 1.232419497, 1.465093153, 1.73845905, 2.057535651, 2.426627838, 2.848393205,
    3.322567571, 3.844480682, 4.403743593, 4.983764322, 5.562817273, 6.116969323, 6.624248234,
    7.068579121, 7.468205581, 7.776783604, 8.020254048, 8.207467661, 8.348497102, 8.41534887
])
methane_loading_data = np.array([
    0.002663325, 0.003180068, 0.003796965, 0.004533391, 0.005412456, 0.006461719, 0.007714043,
    0.009208602, 0.010992082, 0.013120111, 0.015658945, 0.01868747, 0.022299574, 0.026606942,
    0.031742359, 0.037863593, 0.04515796, 0.053847668, 0.064196054, 0.076514841, 0.091172523,
    0.108604022, 0.129321686, 0.153927728, 0.183128073, 0.217747521, 0.251835889, 0.303831956,
    0.361289257, 0.4291802, 0.509198748, 0.603220533, 0.713276867, 0.841501475, 0.990039283,
    1.160905546, 1.355784922, 1.575766198, 1.821022119, 2.090467637, 2.381462264, 2.689653258,
    3.009064794, 3.265608328, 3.575998113, 3.871724227, 4.146318367, 4.524296217, 4.706734644
])

consensus_df = pd.DataFrame({
    "Pressure (Pa)": pressure_pascal_data,
    "CO2 Loading (mol/kg)": co2_loading_data,
    "CH4 Loading (mol/kg)": methane_loading_data
})

# MAX_PHYSICAL_QM_CO2 = 14.31  # Maximum physical loading for CO2 from consensus data


# --- 3. FIT TOTH MODEL USING SCIPY (ALTERNATIVE TO RUPTURA FITTING) ---
print("⚙️  Fitting Toth models and reporting fit quality...")

def toth_model(P, Qm, Kt, t):
    """The Toth isotherm model equation."""
    return (Qm * Kt * P) / (1 + (Kt * P)**t)**(1/t)

def calculate_gof_metrics(y_true, y_pred):
    """Calculates RMSE and R-squared for goodness of fit."""
    residuals = y_true - y_pred
    # RMSE: Root Mean Squared Error. Lower is better. Same units as data.
    rmse = np.sqrt(np.nanmean(residuals**2)) # Use nanmean to handle NaNs
    # R²: Coefficient of determination. Closer to 1.0 is better.
    # Ensure y_true and y_pred are filtered for NaNs before R2 calculation
    valid_indices = ~np.isnan(y_true) & ~np.isnan(y_pred)
    if np.sum(valid_indices) < 2: # Need at least 2 points for R2
        return rmse, np.nan # Return NaN if not enough valid points
    ss_res = np.sum(residuals[valid_indices]**2)
    ss_tot = np.sum((y_true[valid_indices] - np.nanmean(y_true))**2)
    
    if ss_tot == 0: # Avoid division by zero if all true values are the same
        return rmse, 1.0 if ss_res == 0 else 0.0 # R2 is 1 if perfect fit, 0 otherwise
    
    r_squared = 1 - (ss_res / ss_tot)
    return rmse, r_squared

MAX_PHYSICAL_QM_CO2 = 8.5  # Maximum physical loading for CO2 from consensus data
MAX_PHYSICAL_QM_CH4 = 6  # Maximum physical loading

def fit_toth_model_scipy(gas_name, pressure_data, loading_data, max_physical_Qm):
    """Fit Toth model with bounds informed by literature and experimental data."""
    print(f"    -> Fitting Toth model for {gas_name}...")
    
    valid_data_indices = ~np.isnan(loading_data)
    pressure_data_clean = pressure_data[valid_data_indices]
    loading_data_clean = loading_data[valid_data_indices]
    
    # --- THIS IS THE CORRECTED LINE ---
    # Use max_physical_Qm as the initial guess to ensure it's within the bounds
    initial_Qm = max_physical_Qm 
    # --- END CORRECTION ---

    initial_Kt = 0.1 / np.min(pressure_data_clean) if np.min(pressure_data_clean) > 0 else 1e-5
    initial_t = 0.8
    
    # Set the recommended bounds from our previous discussion
    # bounds = ([max_physical_Qm * 0.9, 1e-12, 0.8],    # Lower bounds [Qm, Kt, t]
    #           [max_physical_Qm * 1.6, np.inf, 0.99])   # Upper bounds [Qm, Kt, t]
    
    bounds = ([max_physical_Qm * 0.7, 1e-12, 0.8],    # Lower bounds [Qm, Kt, t]
              [max_physical_Qm * 1.8,  np.inf, 0.99])   # Upper bounds [Qm, Kt, t]
    p0 = [initial_Qm, initial_Kt, initial_t]
    
    try:
        popt, pcov = curve_fit(toth_model, pressure_data_clean, loading_data_clean, 
                               p0=p0, bounds=bounds, maxfev=10000)
        
        fitted_Qm, fitted_Kt, fitted_t = popt
        
        predicted_loading = toth_model(pressure_data_clean, *popt)
        rmse, r_squared = calculate_gof_metrics(loading_data_clean, predicted_loading)
        
        print(f" {gas_name} Fit Complete.")
        print(f"       - Parameters: Qm={fitted_Qm:.4f}, Kt={fitted_Kt:.2e}, t={fitted_t:.4f}")
        print(f"       - Fit Quality: R² = {r_squared:.6f}, RMSE = {rmse:.6f}")
        
        return fitted_Qm, fitted_Kt, fitted_t
        
    except Exception as e:
        print(f" {gas_name} Fit Failed: {str(e)}")
        return max_physical_Qm, 1e-5, 1.0

# --- Fit the models ---
fitted_Qm_co2, fitted_Kt_co2, fitted_t_co2 = fit_toth_model_scipy("CarbonDioxide", pressure_pascal_data, co2_loading_data, MAX_PHYSICAL_QM_CO2)
fitted_Qm_ch4, fitted_Kt_ch4, fitted_t_ch4 = fit_toth_model_scipy("Methane", pressure_pascal_data, methane_loading_data, MAX_PHYSICAL_QM_CH4)


# <<< CORRECTED run_iast_parameter_sweep FUNCTION >>>
def run_iast_parameter_sweep(co2_fractions, pressures_pa):
    all_results = []
    total_calculations = len(co2_fractions) * len(pressures_pa)
    print(f"\n Starting IAST parameter sweep for {total_calculations} points...")
    
    calculation_count = 0
    # Outer loop for gas composition
    for i, y_co2 in enumerate(co2_fractions):
        y_ch4 = 1.0 - y_co2
        
        # Inner loop for pressure
        for p_pa in pressures_pa:
            calculation_count += 1
            print(f"\r   Calculating... Point {calculation_count}/{total_calculations} (y_CO2={y_co2:.2f}, P={p_pa/1e5:.2f} bar)", end="")
            
            # --- MOVED ---
            # Define the components object INSIDE the inner loop
            # This ensures a completely fresh state for every single calculation point.
            components = ruptura.Components([
                {"MoleculeName": "CarbonDioxide", "GasPhaseMolFraction": y_co2, "isotherms": [["Toth", fitted_Qm_co2, fitted_Kt_co2, fitted_t_co2]]},
                {"MoleculeName": "Methane", "GasPhaseMolFraction": y_ch4, "isotherms": [["Toth", fitted_Qm_ch4, fitted_Kt_ch4, fitted_t_ch4]]},
            ])
            # --- END MOVE ---

            try:
                iast_results_3d = ruptura.MixturePrediction(
                    components=components,
                    PressureStart=p_pa, PressureEnd=p_pa, NumberOfPressurePoints=1,
                    # MixturePredictionMethod="IAST"
                    MixturePredictionMethod="IAST"
                ).compute()
                
                df_point = process_ruptura_3d_output_corrected(iast_results_3d, y_co2, y_ch4)
                df_point['Feed_yCO2'] = y_co2
                df_point['Feed_yCH4'] = y_ch4
                all_results.append(df_point)
                
            except Exception as e:
                print(f"\n  Error at y_CO2={y_co2:.2f}, P={p_pa/1e5:.1f} bar: {str(e)}")
                # Handle error case
                df_point = pd.DataFrame({
                    'Total Pressure [Pa]': [p_pa],
                    'Loading CO2 [mol/kg]': [np.nan], 'Loading CH4 [mol/kg]': [np.nan],
                    'Adsorbed_xCO2': [np.nan], 'Adsorbed_xCH4': [np.nan],
                    'Selectivity_CO2_CH4': [np.nan],
                    'Feed_yCO2': [y_co2], 'Feed_yCH4': [y_ch4]
                })
                all_results.append(df_point)
            
    print("\n IAST sweep complete.")
    return pd.concat(all_results, ignore_index=True)



# <<< FINAL AND COMPLETE FUNCTION >>>
def process_ruptura_3d_output_corrected(ruptura_iast_data, y_co2_feed, y_ch4_feed):
    """
    Correctly processes the (1, 2, 6) output array from ruptura,
    works around the library bug, and creates the DataFrame correctly.
    """
    # --- Step 1: Extract the correct scalar values from the raw array ---
    pressure = ruptura_iast_data[0, 0, 0]   # Pressure from index 0
    x_co2_ads = ruptura_iast_data[0, 0, 4]    # Adsorbed Mole Fraction (x) from index 4
    x_ch4_ads = ruptura_iast_data[0, 1, 4]
    q_total = ruptura_iast_data[0, 0, 5]      # Total Mixture Loading (qt) from index 5

    # --- Step 2: Manually calculate the TRUE component loadings ---
    loading_co2 = x_co2_ads * q_total
    loading_ch4 = x_ch4_ads * q_total

    # --- Step 3: Calculate selectivity ---
    selectivity = np.nan
    if y_ch4_feed > 1e-12 and x_ch4_ads > 1e-12 and y_co2_feed > 1e-12:
        selectivity = (x_co2_ads / y_co2_feed) / (x_ch4_ads / y_ch4_feed)

    # --- Step 4: Return the DataFrame, wrapping all values in lists [] ---
    return pd.DataFrame({
        'Total Pressure [Pa]': [pressure],
        'Loading CO2 [mol/kg]': [loading_co2],
        'Loading CH4 [mol/kg]': [loading_ch4],
        'Adsorbed_xCO2': [x_co2_ads],
        'Adsorbed_xCH4': [x_ch4_ads],
        'Selectivity_CO2_CH4': [selectivity]
    })




sweep_results_df = run_iast_parameter_sweep(FEED_CO2_FRACTIONS, PRESSURE_SWEEP_PA)
sweep_results_df['Total Pressure [bar]'] = sweep_results_df['Total Pressure [Pa]'] / 1e5
sweep_results_df['partial_pressure_CO2 [bar]'] = sweep_results_df['Total Pressure [bar]'] * sweep_results_df['Feed_yCO2']
sweep_results_df['partial_pressure_CH4 [bar]'] = sweep_results_df['Total Pressure [bar]'] * sweep_results_df['Feed_yCH4']

# Rearranging columns to match GRAPHIAST format
sweep_results_df = sweep_results_df[['Total Pressure [bar]','Feed_yCO2','Feed_yCH4','partial_pressure_CO2 [bar]','partial_pressure_CH4 [bar]','Loading CO2 [mol/kg]','Loading CH4 [mol/kg]','Selectivity_CO2_CH4',
                                      'Adsorbed_xCO2','Adsorbed_xCH4','Total Pressure [Pa]']]

# --- 5. EXPORT RESULTS TO CSV ---
csv_path = os.path.join(OUTPUT_DIR, "iast_parameter_sweep_results.csv")
sweep_results_df.to_csv(csv_path, index=False)
print(f"\n Full results saved to: {csv_path}")

# --- 6. PLOTTING ---
def plot_loadings_with_experimental_data(df_equimolar, df_experimental, output_dir):
    print("\n Generating loading comparison plot...")
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Filter out NaN values for plotting
    valid_equimolar = df_equimolar.dropna(subset=['Loading CO2 [mol/kg]', 'Loading CH4 [mol/kg]'])
    
    if len(valid_equimolar) > 0:
        ax.semilogx(valid_equimolar['Total Pressure [Pa]'], valid_equimolar['Loading CO2 [mol/kg]'], 
                   'o-', ms=4, label='IAST CO₂ (50/50 Mix)', color='red')
        ax.semilogx(valid_equimolar['Total Pressure [Pa]'], valid_equimolar['Loading CH4 [mol/kg]'], 
                   's-', ms=4, label='IAST CH₄ (50/50 Mix)', color='blue')
    
    ax.scatter(df_experimental['Pressure (Pa)'], df_experimental['CO2 Loading (mol/kg)'],
               facecolors='none', edgecolors='red', marker='o', label='Expt. Pure CO₂')
    ax.scatter(df_experimental['Pressure (Pa)'], df_experimental['CH4 Loading (mol/kg)'],
               facecolors='none', edgecolors='blue', marker='s', label='Expt. Pure CH₄')
    ax.set_xlabel('Total Pressure (Pa)')
    ax.set_ylabel('Loading (mol/kg)')
    ax.set_title('IAST Mixture vs. Pure Component Experimental Loadings')
    ax.legend()
    ax.grid(True, which="both", ls="--", alpha=0.5)
    plt.tight_layout()
    plot_path = os.path.join(output_dir, "loadings_comparison_plot.png")
    plt.savefig(plot_path, dpi=300)
    plt.show()
    print(f"   -> Plot saved to: {plot_path}")

def plot_selectivity_surface(df, output_dir):
    print(" Generating selectivity surface plot...")
    
    # Filter out NaN values
    df_clean = df.dropna(subset=['Selectivity_CO2_CH4'])
    
    if len(df_clean) == 0:
        print("  No valid selectivity data for surface plot")
        return
    
    pivot_df = df_clean.pivot_table(index='Feed_yCO2', columns='Total Pressure [bar]', values='Selectivity_CO2_CH4')
    
    if pivot_df.empty:
        print("  Empty pivot table for selectivity surface")
        return
    
    X, Y = np.meshgrid(pivot_df.columns, pivot_df.index)
    Z = pivot_df.values
    
    # Handle any remaining NaN values
    Z_masked = np.ma.masked_invalid(Z)
    
    fig, ax = plt.subplots(figsize=(10, 7))
    contour = ax.pcolormesh(X, Y, Z_masked, shading='auto', cmap='viridis')
    cbar = fig.colorbar(contour)
    cbar.set_label('Selectivity ($\\alpha_{CO_2/CH_4}$)')
    
    # Only add contour lines if there are valid values
    if not np.all(np.isnan(Z)):
        contour_lines = ax.contour(X, Y, Z_masked, colors='white', linewidths=0.5)
        ax.clabel(contour_lines, inline=True, fontsize=8, fmt='%1.1f')
    
    ax.set_xlabel('Total Pressure (bar)')
    ax.set_ylabel('CO₂ Mole Fraction in Feed ($y_{CO_2}$)')
    ax.set_title('CO₂/CH₄ Selectivity Surface')
    ax.set_xscale('log')
    plt.tight_layout()
    plot_path = os.path.join(output_dir, "selectivity_surface_plot.png")
    plt.savefig(plot_path, dpi=300)
    plt.show()
    print(f"   -> Plot saved to: {plot_path}")

def plot_selectivity_lines_vs_composition(df, output_dir):
    print(" Generating selectivity line plot...")
    
    # Filter out NaN values
    df_clean = df.dropna(subset=['Selectivity_CO2_CH4'])
    
    if len(df_clean) == 0:
        print("  No valid selectivity data for line plot")
        return
    
    fig, ax = plt.subplots(figsize=(9, 6))
    pressures_to_plot_bar = [1.0, 10.0, 30.0, 50.0]
    
    for p_bar in pressures_to_plot_bar:
        available_pressures = df_clean['Total Pressure [bar]'].unique()
        if len(available_pressures) == 0:
            continue
        actual_pressure = available_pressures[np.abs(available_pressures - p_bar).argmin()]
        pressure_data = df_clean[df_clean['Total Pressure [bar]'] == actual_pressure]
        
        if len(pressure_data) > 0:
            ax.plot(pressure_data['Feed_yCO2'], pressure_data['Selectivity_CO2_CH4'], 'o-', ms=4,
                    label=f'{actual_pressure:.0f} bar')

    ax.set_xlabel('CO₂ Mole Fraction in Feed ($y_{CO_2}$)')
    ax.set_ylabel('Selectivity ($\\alpha_{CO_2/CH_4}$)')
    ax.set_title('Selectivity vs. Feed Composition at Various Pressures')
    ax.legend(title="Total Pressure")
    ax.grid(True, which="both", ls="--", alpha=0.5)
    ax.set_xlim(0, 1)
    plt.tight_layout()
    plot_path = os.path.join(output_dir, "selectivity_lines_plot.png")
    plt.savefig(plot_path, dpi=300)
    plt.show()
    print(f"   -> Plot saved to: {plot_path}")

# Generate plots
equimolar_results = sweep_results_df[sweep_results_df['Feed_yCO2'] == 0.5]
plot_loadings_with_experimental_data(equimolar_results, consensus_df, OUTPUT_DIR)
plot_selectivity_surface(sweep_results_df, OUTPUT_DIR)
plot_selectivity_lines_vs_composition(sweep_results_df, OUTPUT_DIR)

print("\n All tasks completed!")

## Section 6: IAST Model Comparison Against Binary Experimental Data

This section loads pre-computed IAST results from multiple adsorption models (Toth, Langmuir, Quadratic, and a spline Interpolator) alongside experimental binary mixture adsorption data for ZIF-8 CO2/CH4. Goodness-of-fit metrics (RMSE and R-squared) are calculated for each model against the experimental loadings and selectivity, providing a quantitative basis for model selection.

In [ ]:
# Load the new CSV data
# df = pd.read_csv('combo data.csv')
df = pd.read_csv('./Results/graphIAST resuults/exact binary pressure/combo data.csv')

# Clean column names more robustly:
# 1. Strip leading/trailing whitespace
# 2. Replace multiple spaces with a single space
df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

# --- 2. CONVERT EXPERIMENTAL UNITS to mol/kg ---
# Assuming 'Binary Exp-q1' and 'Binary Exp-q2' are the experimental loadings
# and are numerically equivalent to mol/kg or mmol/g.
df['Exp_qCO2_mol_kg'] = df['Binary Exp-q1'] * 1.0
df['Exp_qCH4_mol_kg'] = df['Binary Exp-q2'] * 1.0
df['Exp_Selectivity_CO2_CH4'] = df['Binary Exp- selectivity']
print(" Experimental loadings and selectivity identified and prepared.")

# --- 3. CALCULATE RMSE AND R² FOR EACH MODEL ---
print("\n Calculating Goodness-of-Fit Metrics (RMSE & R²)...")

metrics = {}

# Print cleaned column names for debugging
print("Cleaned DataFrame columns:", df.columns.tolist())

models_to_compare = {
    'Toth': {
        'qCO2_pred': df['Toth -Loading CO2 [mol/kg]'],
        'qCH4_pred': df['Toth -Loading CH4 [mol/kg]'],
        'selectivity_pred': df['Toth -Selectivity_CO2_CH4']
    },
    'Interpolator': {
        'qCO2_pred': df['interpolator - loading Gas 1 [mmol/g]'],
        'qCH4_pred': df['interpolator - loading Gas 2 [mmol/g]'],
        'selectivity_pred': df['interpolator Selectivity']
    },
    'Langmuir': {
        'qCO2_pred': df['langmuir - loading Gas 1 [mmol/g]'],
        'qCH4_pred': df['langmuir loading Gas 2 [mmol/g]'],
        'selectivity_pred': df['langmuir - Selectivity']
    },
    'Quadratic': {
        'qCO2_pred': df['quadratic- loading Gas 1 [mmol/g]'],
        'qCH4_pred': df['quadratic- loading Gas 2 [mmol/g]'],
        'selectivity_pred': df['quadratic-Selectivity']
    }
}

for model_name, preds in models_to_compare.items():
    metrics[model_name] = {}

    # CO2 Loading
    rmse_co2 = np.sqrt(mean_squared_error(df['Exp_qCO2_mol_kg'], preds['qCO2_pred']))
    r2_co2 = r2_score(df['Exp_qCO2_mol_kg'], preds['qCO2_pred'])
    metrics[model_name]['RMSE_CO2_Loading'] = rmse_co2
    metrics[model_name]['R2_CO2_Loading'] = r2_co2

    # CH4 Loading
    rmse_ch4 = np.sqrt(mean_squared_error(df['Exp_qCH4_mol_kg'], preds['qCH4_pred']))
    r2_ch4 = r2_score(df['Exp_qCH4_mol_kg'], preds['qCH4_pred'])
    metrics[model_name]['RMSE_CH4_Loading'] = rmse_ch4
    metrics[model_name]['R2_CH4_Loading'] = r2_ch4

    # Selectivity
    rmse_selectivity = np.sqrt(mean_squared_error(df['Exp_Selectivity_CO2_CH4'], preds['selectivity_pred']))
    r2_selectivity = r2_score(df['Exp_Selectivity_CO2_CH4'], preds['selectivity_pred'])
    metrics[model_name]['RMSE_Selectivity'] = rmse_selectivity
    metrics[model_name]['R2_Selectivity'] = r2_selectivity

# Display Results
metrics_df = pd.DataFrame(metrics).T # Transpose for better readability
print("\n=== Model Comparison Metrics (RMSE & R²) vs. Experimental Data ===")
print(metrics_df.to_string(float_format="%.4f"))

# --- 4. INTERPRETATION AND INSIGHTS ---
print("\n=== Interpretation of Results ===")

# Interpretation of Loadings
print("\n--- Loadings (CO2 and CH4) ---")
print("The RMSE values for CO2 and CH4 loadings indicate how close the predicted loadings are to the experimental ones. Lower RMSE means better agreement.")
print("R² values closer to 1.0 indicate that the model explains more of the variability in the experimental loadings.")
print("Observation:")
print(metrics_df[['RMSE_CO2_Loading', 'R2_CO2_Loading', 'RMSE_CH4_Loading', 'R2_CH4_Loading']].to_string(float_format="%.4f"))
print("\nInsights:")
print("- **Tóth Model (RUPTURA):** Shows the lowest RMSE and highest R² for both CO2 and CH4 loadings. This means Tóth provides the most accurate predictions for the actual amounts of both gases adsorbed in the mixture compared to experimental data.")
print("- **Interpolator Model:** Also performs very well for loadings, with low RMSE and high R², indicating it closely follows the experimental loading behavior.")
print("- **Langmuir and Quadratic Models:** Exhibit higher RMSE and lower R² for loadings, indicating they are less accurate in predicting the adsorbed amounts compared to Tóth and Interpolator.")
print("Conclusion for Loadings: The Tóth model is the most accurate for predicting individual component loadings in the mixture, followed closely by the Interpolator.")

# Interpretation of Selectivity
print("\n--- Selectivity ---")
print("The RMSE for selectivity indicates how well the predicted selectivity matches the experimental selectivity. Lower RMSE means better agreement.")
print("R² for selectivity shows how much of the experimental selectivity's variance is explained by the model.")
print("Observation:")
print(metrics_df[['RMSE_Selectivity', 'R2_Selectivity']].to_string(float_format="%.4f"))
print("\nInsights:")
print("- **Tóth Model (RUPTURA):** Shows the lowest RMSE and highest R² for selectivity. This means Tóth provides the best overall prediction of selectivity compared to experimental data, even if it misses the exact non-monotonic trend.")
print("- **Interpolator Model:** Also performs well for selectivity, capturing some of the non-monotonic behavior, but Tóth still has a slightly lower RMSE.")
print("- **Langmuir and Quadratic Models:** Show higher RMSE and lower R² for selectivity, indicating poorer agreement with experimental selectivity.")
print("Conclusion for Selectivity: The Tóth model provides the most accurate selectivity predictions against experimental data.")

# Overall Conclusion
print("\n--- Overall Conclusion ---")
print("This quantitative comparison confirms that the **Tóth model, as fitted and used in RUPTURA, provides the most accurate IAST predictions** for both individual component loadings and overall selectivity when compared against experimental mixture adsorption data for ZIF-8 CO2/CH4.")
print("The differences in selectivity trends (e.g., Tóth's smoother trend vs. experimental non-monotonicity) highlight the inherent complexities of mixture adsorption and the limitations of IAST's 'ideal solution' assumption, even with a robust pure-component model. However, the quantitative agreement is strong.")

## Section 7: Visualisation

This section generates final publication-quality figures. Two types of plots are produced:

1. IAST mixture loading plots — CO2 and CH4 adsorbed loadings as a function of feed composition at selected total pressures (1, 10, 20, 50 bar)
2. Consensus isotherm panel — Combined four-panel figure showing full-range and zoomed (0.01-1 bar) adsorption isotherms for CO2 and CH4 with consensus mean and confidence interval overlays

### 7.1 IAST Mixture Loadings by Pressure

In [ ]:
# Create the data
data = {
    'yco2': [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 0.75, 0.8, 0.9, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99],
    
    # 1 Bar data
    'CO2_1bar': [0.0099, 0.0198, 0.0297, 0.0395, 0.0494, 0.0592, 0.0690, 0.0788, 0.0886, 0.0984, 0.1955, 0.2436, 0.2913, 0.3388, 0.3860, 0.4330, 0.4796, 0.5260, 0.5721, 0.6635, 0.7088, 0.7538, 0.8432, 0.8520, 0.8609, 0.8698, 0.8786, 0.8874, 0.8963, 0.9051, 0.9139, 0.9227],
    'CH4_1bar': [0.2770, 0.2740, 0.2709, 0.2679, 0.2649, 0.2618, 0.2588, 0.2558, 0.2528, 0.2498, 0.2202, 0.2056, 0.1911, 0.1767, 0.1625, 0.1484, 0.1344, 0.1205, 0.1067, 0.0794, 0.0659, 0.0525, 0.0261, 0.0234, 0.0208, 0.0182, 0.0156, 0.0130, 0.0104, 0.0078, 0.0052, 0.0026],
    
    # 10 Bar data
    'CO2_10bar': [0.0772, 0.1536, 0.2294, 0.3045, 0.3789, 0.4527, 0.5258, 0.5983, 0.6701, 0.7413, 1.4204, 1.7388, 2.0442, 2.3372, 2.6185, 2.8888, 3.1485, 3.3982, 3.6384, 4.0920, 4.3063, 4.5128, 4.9037, 4.9412, 4.9785, 5.0155, 5.0523, 5.0887, 5.1250, 5.1609, 5.1967, 5.2321],
    'CH4_10bar': [2.0389, 2.0062, 1.9738, 1.9418, 1.9102, 1.8789, 1.8479, 1.8173, 1.7870, 1.7571, 1.4746, 1.3442, 1.2204, 1.1028, 0.9910, 0.8846, 0.7835, 0.6872, 0.5955, 0.4249, 0.3455, 0.2698, 0.1287, 0.1153, 0.1020, 0.0888, 0.0758, 0.0628, 0.0500, 0.0374, 0.0248, 0.0123],
    
    # 20 Bar data
    'CO2_20bar': [0.1267, 0.2515, 0.3746, 0.4960, 0.6157, 0.7337, 0.8500, 0.9648, 1.0779, 1.1895, 2.2257, 2.6941, 3.1328, 3.5442, 3.9303, 4.2931, 4.6343, 4.9555, 5.2581, 5.8126, 6.0669, 6.3073, 6.7500, 6.7916, 6.8329, 6.8736, 6.9140, 6.9539, 6.9934, 7.0325, 7.0711, 7.1094],
    'CH4_20bar': [3.1766, 3.1138, 3.0522, 2.9915, 2.9319, 2.8733, 2.8157, 2.7590, 2.7033, 2.6485, 2.1471, 1.9251, 1.7199, 1.5302, 1.3544, 1.1915, 1.0404, 0.9000, 0.7695, 0.5351, 0.4299, 0.3317, 0.1546, 0.1382, 0.1220, 0.1060, 0.0902, 0.0747, 0.0593, 0.0442, 0.0293, 0.0145],
    
    # 35 Bar data
    'CO2_35bar': [0.1729, 0.3425, 0.5089, 0.6723, 0.8326, 0.9900, 1.1445, 1.2961, 1.4451, 1.5913, 2.9169, 3.4967, 4.0285, 4.5170, 4.9663, 5.3803, 5.7621, 6.1149, 6.4412, 7.0238, 7.2841, 7.5262, 7.9615, 8.0018, 8.0415, 8.0807, 8.1193, 8.1574, 8.1950, 8.2321, 8.2687, 8.3048],
    'CH4_35bar': [4.0881, 3.9933, 3.9004, 3.8097, 3.7208, 3.6340, 3.5489, 3.4658, 3.3844, 3.3047, 2.5947, 2.2915, 2.0178, 1.7703, 1.5461, 1.3427, 1.1579, 0.9899, 0.8367, 0.5695, 0.4528, 0.3460, 0.1583, 0.1413, 0.1245, 0.1080, 0.0918, 0.0758, 0.0602, 0.0447, 0.0296, 0.0147],
    
    # 50 Bar data
    'CO2_50bar': [0.2068, 0.4092, 0.6071, 0.8007, 0.9901, 1.1755, 1.3568, 1.5343, 1.7080, 1.8780, 3.3933, 4.0404, 4.6249, 5.1536, 5.6329, 6.0681, 6.4640, 6.8248, 7.1542, 7.7313, 7.9844, 8.2171, 8.6287, 8.6663, 8.7034, 8.7398, 8.7757, 8.8110, 8.8458, 8.8800, 8.9138, 8.9469],
    'CH4_50bar': [4.6556, 4.5355, 4.4185, 4.3045, 4.1933, 4.0849, 3.9792, 3.8762, 3.7757, 3.6777, 2.8198, 2.4629, 2.1459, 1.8637, 1.6120, 1.3872, 1.1859, 1.0053, 0.8431, 0.5655, 0.4466, 0.3391, 0.1533, 0.1366, 0.1203, 0.1042, 0.0885, 0.0730, 0.0579, 0.0430, 0.0284, 0.0141]
}

# Create DataFrame
df = pd.DataFrame(data)

# Set up the plot with a modern style
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Define colors and markers for CH4 and CO2
ch4_color = '#1f77b4'
co2_color = '#ff7f0e'
pressures = [1, 10, 20, 50]  # Removed 35 bar

# Plot each pressure in its own subplot
for i, pressure in enumerate(pressures):
    ax = axes[i]
    
    # Plot CH4 (filled circles)
    ax.scatter(df['yco2'], df[f'CH4_{pressure}bar'], 
               color=ch4_color, marker='o', s=60, alpha=0.8,
               label='CH$_4$', edgecolors='white', linewidth=0.5)
    
    # Plot CO2 (filled triangles)
    ax.scatter(df['yco2'], df[f'CO2_{pressure}bar'], 
               color=co2_color, marker='^', s=60, alpha=0.8,
               label='CO$_2$', edgecolors='white', linewidth=0.5)
    
    ax.set_xlabel('Feed CO$_2$ Mole Fraction', fontsize=11, fontweight='bold')
    ax.set_ylabel('Loading (mmol/g)', fontsize=11, fontweight='bold')
    ax.text(0.02, 0.98, f'{pressure} Bar', transform=ax.transAxes, 
            fontsize=12, fontweight='bold', verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.legend(fontsize=10, frameon=True, fancybox=True)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.02, 1.02)

# Adjust layout
plt.tight_layout()
plt.show()

# Print caption suggestion
print("\nSUGGESTED FIGURE CAPTION:")
print("IAST loading isotherms for CH₄ (circles) and CO₂ (triangles) as a function of CO₂ mole fraction at different total pressures (1, 10, 20, and 50 bar). The plots demonstrate the competitive adsorption behavior where CH₄ loading increases with CO₂ mole fraction while CO₂ loading decreases, indicating preferential adsorption of CO₂ over CH₄. Higher total pressures result in increased loading capacities for both gases.")

### 7.2 Consensus Isotherm Panel (CO2 and CH4)

In [ ]:
# === Config ===
filename = "./5_co2methaneconsensus.xlsx"
pressure_threshold = 17.3147153990574
output_dir = "./plots"
os.makedirs(output_dir, exist_ok=True)

discarded_map = {
 'co2': ["10.1021acs.jpcc.5b01519.Isotherm18.json"],
 'ch4': ["10.1016j.seppur.2019.115850.Isotherm8.json"]
}

# Global font updates
plt.rcParams.update({
 'font.size': 14,
 'axes.labelsize': 14,
 'xtick.labelsize': 12,
 'ytick.labelsize': 12,
 'legend.fontsize': 11
})

# === Plot Function (takes external ax) ===
def plot_isotherms(ax, df, pressure, consensus_mean, ci_lower, ci_upper, isotherm_cols,
 gas_label, zoom=False, show_ylabel=True, show_xlabel=True, consensus_thin=False):

 error_lower = consensus_mean - ci_lower
 error_upper = ci_upper - consensus_mean
 mask_error = pressure <= pressure_threshold

 cmap = plt.get_cmap('tab20')
 color_cycle = [cmap(i) for i in np.linspace(0, 1, len(isotherm_cols))]

 for idx, col in enumerate(isotherm_cols):
 is_discarded = col in discarded_map.get(gas_label, [])
 label = col.replace('.json', '')
 linestyle = ':' if is_discarded else '--'
 if is_discarded:
 label = f"DISCARDED – {label}"
 color = 'crimson' # Red for discarded
 else:
 color = color_cycle[idx % len(color_cycle)]

 ax.plot(
 pressure, df[col],
 linestyle=linestyle,
 linewidth=2.2,
 alpha=0.95,
 label=label,
 color=color
 )

 consensus_linewidth = 1.5 if consensus_thin else 2.5
 ax.plot(
 pressure, consensus_mean,
 'o-', color='black',
 linewidth=consensus_linewidth,
 markersize=3,
 label='Consensus Mean',
 zorder=5
 )

 ax.errorbar(
 pressure[mask_error], consensus_mean[mask_error],
 yerr=[error_lower[mask_error], error_upper[mask_error]],
 fmt='none', ecolor='gray', elinewidth=1.5, capsize=4,
 label=f'Consensus CI (≤ {pressure_threshold:.2f} bar)', zorder=4
 )

 if show_xlabel:
 ax.set_xlabel("Pressure [bar]")
 else:
 ax.set_xlabel("")

 if show_ylabel:
 ax.set_ylabel("Uptake [mmol/g]")
 else:
 ax.set_ylabel("")

 ax.grid(True, which='both', linestyle='--', linewidth=0.6)

 ax.legend(
 loc="best",
 fontsize=11,
 frameon=True,
 framealpha=0.1, # semi-transparent
 borderpad=0.8,
 handlelength=2
 )

 if zoom:
 ax.set_xlim(0.01, 1.05)
 yaxis_max = 0.35 if gas_label == 'ch4' else 0.85
 ax.set_ylim(0.01, yaxis_max)
 ax.set_xscale('linear')
 ax.set_yticks(np.linspace(0.01, yaxis_max, 6))
 else:
 ax.set_xscale('log')

 # === Individual subplot export ===
individual_labels = ['A', 'B', 'C', 'D']
plot_idx = 0
for gas_label in ['co2', 'ch4']:
 df = pd.read_excel(filename, sheet_name=gas_label)
 df = df.dropna(subset=["Pressure (bar)"])

 pressure = df["Pressure (bar)"]
 consensus_mean = df["Consensus_Mean_mmol_g"]
 ci_lower = df["Consensus_CI_Lower_mmol_g"]
 ci_upper = df["Consensus_CI_Upper_mmol_g"]
 isotherm_cols = [col for col in df.columns if col.endswith('.json')]

 # A or C — full range
 fig, ax = plt.subplots(figsize=(10, 6))
 plot_isotherms(
 ax, df, pressure, consensus_mean, ci_lower, ci_upper, isotherm_cols,
 gas_label,
 zoom=False,
 show_ylabel=True,
 show_xlabel=True,
 consensus_thin=True
 )
 ax.text(0.98, 0.02, individual_labels[plot_idx], transform=ax.transAxes,
 fontsize=16, fontweight='bold', verticalalignment='bottom', horizontalalignment='right')
 save_path = os.path.join(output_dir, f"{gas_label}_full_{individual_labels[plot_idx]}.png")
 plt.tight_layout()
 plt.savefig(save_path, dpi=400)
 plt.close()
 print(f"Saved: {save_path}")
 plot_idx += 1

 # B or D — zoomed
 fig, ax = plt.subplots(figsize=(10, 6))
 plot_isotherms(
 ax, df, pressure, consensus_mean, ci_lower, ci_upper, isotherm_cols,
 gas_label,
 zoom=True,
 show_ylabel=True,
 show_xlabel=True,
 consensus_thin=False
 )
 ax.text(0.98, 0.02, individual_labels[plot_idx], transform=ax.transAxes,
 fontsize=16, fontweight='bold', verticalalignment='bottom', horizontalalignment='right')
 save_path = os.path.join(output_dir, f"{gas_label}_zoom_{individual_labels[plot_idx]}.png")
 plt.tight_layout()
 plt.savefig(save_path, dpi=400)
 plt.close()
 print(f"Saved: {save_path}")
 plot_idx += 1


# === Subplot Labels (bottom-right positions) ===
subplot_labels = ['A', 'B', 'C', 'D']

# === Create Combined Figure ===
fig, axs = plt.subplots(2, 2, figsize=(18, 12))
axs = axs.flatten()

plot_idx = 0
for gas_label in ['co2', 'ch4']:
 df = pd.read_excel(filename, sheet_name=gas_label)
 df = df.dropna(subset=["Pressure (bar)"])

 pressure = df["Pressure (bar)"]
 consensus_mean = df["Consensus_Mean_mmol_g"]
 ci_lower = df["Consensus_CI_Lower_mmol_g"]
 ci_upper = df["Consensus_CI_Upper_mmol_g"]
 isotherm_cols = [col for col in df.columns if col.endswith('.json')]

 # Full range: A and C (no x-label)
 plot_isotherms(
 axs[plot_idx],
 df, pressure, consensus_mean, ci_lower, ci_upper, isotherm_cols,
 gas_label,
 zoom=False,
 show_ylabel=True,
 show_xlabel=True, # omit x-axis label
 consensus_thin=True
 )

 # Zoom: B and D (no y-label)
 plot_isotherms(
 axs[plot_idx + 1],
 df, pressure, consensus_mean, ci_lower, ci_upper, isotherm_cols,
 gas_label,
 zoom=True,
 show_ylabel=False, # omit y-axis label
 show_xlabel=True,
 consensus_thin=False
 )

 plot_idx += 2

# === Add subplot tags (bottom-right inside plot) ===
for idx, ax in enumerate(axs):
 ax.text(0.98, 0.02, subplot_labels[idx],
 transform=ax.transAxes,
 fontsize=16, fontweight='bold',
 verticalalignment='bottom', horizontalalignment='right')

plt.tight_layout()
output_path = os.path.join(output_dir, "all_isotherms_combined_labeled_final.png")
plt.savefig(output_path, dpi=400)
plt.show()
print(f"Saved: {output_path}")


### 7.3 Individual Isotherm Set Visualisation Utility

In [ ]:
def plot_single_isotherm_set(ax, df, unique_isotherms, colors, zoom_1bar=False, title_suffix=""):
    """
    Plot isotherms on a given axis
    """
    # Plot each isotherm
    for i, isotherm in enumerate(unique_isotherms):
        # Filter data for this isotherm
        isotherm_data = df[df['isotherm_json'] == isotherm].copy()
        
        # Sort by pressure for proper line plotting
        isotherm_data = isotherm_data.sort_values('pressure')
        
        # Extract data
        pressure = isotherm_data['pressure'].values
        adsorption = isotherm_data['adsorption_mmol_per_g'].values
        
        # Filter for zoom if requested
        if zoom_1bar:
            mask = pressure <= 1.0
            pressure = pressure[mask]
            adsorption = adsorption[mask]
        
        # Use isotherm_json as label (remove .json extension for cleaner look)
        label = isotherm.replace('.json', '')
        
        # Plot with larger markers and thicker lines
        ax.plot(pressure, adsorption, 'o-', 
               color=colors[i], 
               linewidth=2.5, 
               markersize=7,
               label=label,
               alpha=0.85,
               markerfacecolor=colors[i],
               markeredgecolor='white',
               markeredgewidth=0.5)
    
    # Customize the plot
    ax.set_xlabel('Pressure [bar]', fontsize=18, fontweight='bold')
    ax.set_ylabel('Uptake [mmol/g]', fontsize=18, fontweight='bold')
    
    if title_suffix:
        ax.set_title(f'Adsorption Isotherms {title_suffix}', fontsize=20, fontweight='bold', pad=20)
    
    # Clean grid
    ax.grid(True, alpha=0.4, linestyle='-', linewidth=0.8, color='lightgray')
    ax.set_axisbelow(True)
    
    # Position legend outside plot area
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=12,
              frameon=True, framealpha=0.9, edgecolor='gray', 
              borderpad=1, handlelength=2.5)
    
    # Clean tick parameters
    ax.tick_params(axis='both', which='major', labelsize=14, 
                   length=6, width=1.2, color='black')
    
    # Set linear scale
    ax.set_xscale('linear')
    ax.set_yscale('linear')
    
    # Set limits
    if zoom_1bar:
        ax.set_xlim(0, 1.05)
        # Get y range for better scaling
        y_range = ax.get_ylim()
        ax.set_ylim(0, y_range[1] * 1.05)
    else:
        # Add some padding to axes
        x_range = ax.get_xlim()
        y_range = ax.get_ylim()
        x_padding = (x_range[1] - x_range[0]) * 0.02
        y_padding = (y_range[1] - y_range[0]) * 0.02
        ax.set_xlim(max(0, x_range[0] - x_padding), x_range[1] + x_padding)
        ax.set_ylim(max(0, y_range[0] - y_padding), y_range[1] + y_padding)
    
    # Clean spines
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
        spine.set_color('black')

def plot_adsorption_isotherms(csv_file_path, output_dir=None):
    """
    Plot adsorption isotherms from CSV data - creates two plots: full range and 0-1 bar zoom
    
    Parameters:
    csv_file_path (str): Path to the CSV file
    output_dir (str): Optional directory to save plots (if None, will display)
    """
    
    # Read the CSV file
    try:
        df = pd.read_csv(csv_file_path)
        print(f"Successfully loaded {len(df)} data points")
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return
    
    # Clean column names (remove any whitespace)
    df.columns = df.columns.str.strip()
    
    # Get unique isotherms
    unique_isotherms = df['isotherm_json'].unique()
    print(f"Found {len(unique_isotherms)} unique isotherms:")
    for isotherm in unique_isotherms:
        print(f"  - {isotherm}")
    
    # Global font settings for publication quality
    plt.rcParams.update({
        'font.size': 16,
        'axes.labelsize': 18,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'legend.fontsize': 12,
        'axes.linewidth': 1.2,
        'xtick.major.width': 1.2,
        'ytick.major.width': 1.2
    })
    
    # Enhanced color palette with more distinct colors
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_isotherms)))
    
    # Create output directory if specified
    if output_dir:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Plot 1: Full range
    print("\nCreating full range plot...")
    fig1, ax1 = plt.subplots(figsize=(16, 10))
    plot_single_isotherm_set(ax1, df, unique_isotherms, colors, zoom_1bar=False, title_suffix="(Full Range)")
    plt.tight_layout()
    
    if output_dir:
        output_path1 = Path(output_dir) / "isotherms_full_range.png"
        plt.savefig(output_path1, dpi=400, bbox_inches='tight', 
                   facecolor='white', edgecolor='none', format='png', pad_inches=0.2)
        print(f"Full range plot saved to: {output_path1}")
        plt.close()
    else:
        plt.show()
    
    # Plot 2: Zoomed to 0-1 bar
    print("Creating zoomed (0-1 bar) plot...")
    fig2, ax2 = plt.subplots(figsize=(16, 10))
    plot_single_isotherm_set(ax2, df, unique_isotherms, colors, zoom_1bar=True, title_suffix="(0-1 bar)")
    plt.tight_layout()
    
    if output_dir:
        output_path2 = Path(output_dir) / "isotherms_0_to_1_bar.png"
        plt.savefig(output_path2, dpi=400, bbox_inches='tight', 
                   facecolor='white', edgecolor='none', format='png', pad_inches=0.2)
        print(f"Zoomed plot saved to: {output_path2}")
        plt.close()
    else:
        plt.show()
    
    # Print summary statistics
    print("\nSummary:")
    for isotherm in unique_isotherms:
        isotherm_data = df[df['isotherm_json'] == isotherm]
        print(f"{isotherm}: {len(isotherm_data)} data points")
        print(f"  Pressure range: {isotherm_data['pressure'].min():.3f} - {isotherm_data['pressure'].max():.3f} bar")
        print(f"  Adsorption range: {isotherm_data['adsorption_mmol_per_g'].min():.3f} - {isotherm_data['adsorption_mmol_per_g'].max():.3f} mmol/g")
        # Count points in 0-1 bar range
        low_pressure_points = len(isotherm_data[isotherm_data['pressure'] <= 1.0])
        print(f"  Points ≤ 1 bar: {low_pressure_points}")
        print()

def main():
    """
    Main function to run the plotting script
    Modify the file path below to point to your CSV file
    """
    
    # MODIFY THIS PATH TO YOUR CSV FILE
    csv_file_path = "powerpointconfidence.csv"  # Change this to your actual file path
    
    # Optional: specify output path to save the plot
    # output_path = "adsorption_isotherms.png"  # Uncomment and modify to save
    output_path = None  # Set to None to display instead of saving
    
    # Check if file exists
    if not Path(csv_file_path).exists():
        print(f"Error: File '{csv_file_path}' not found!")
        print("Please update the csv_file_path variable with the correct path to your CSV file.")
        return
    
    # Plot the isotherms
    plot_adsorption_isotherms(csv_file_path, output_path)

if __name__ == "__main__":
    main()